# PoliMillionaire â€” All Competitions Notebook
## Competitions covered
| ID | Name |
|----|------|
| 0 | Entertainment |
| 1 | Ancient History & Politics |
| 2 | Science & Nature |
| 3 | Maths |
| 4 | Philosophy & Psychology |
| 5 | News & Current Events |

This notebook merges three independently developed pipelines that all share the same
base model (`Qwen/Qwen2.5-7B-Instruct`) and the same `MillionaireClient` API:

- **Entertainment / Science / Psychology** â€” zero-shot, few-shot, CoT, Wikipedia RAG,
  DuckDuckGo hybrid RAG, multi-model ensemble (from `PoliMillionaire_Starter_Clean`)
- **History** â€” advanced BM25 + sentence-embedding RAG with PyTerrier, cross-encoder
  reranker, direct logit scoring, and agentic tool router (from `History_v2`)
- **News** â€” live Serper news search, Bing RSS fallback + FAISS semantic ranking
  (from `nlpproject_news`)


## 0. Setup â€” Mount Drive & Install Dependencies

In [1]:
from google.colab import drive
drive.mount('/content/gdrive/')


Drive already mounted at /content/gdrive/; to attempt to forcibly remount, call drive.mount("/content/gdrive/", force_remount=True).


In [2]:
import os, sys, importlib.util, subprocess

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# Adjust these paths to match your Drive layout
PACKAGE_PARENT_DIR = '/content/gdrive/MyDrive/Colab Notebooks/NLP_Test'
NLP_ASSIGNMENT_DIR = '/content/gdrive/MyDrive/NLP_assignment'

for _dir in (PACKAGE_PARENT_DIR, NLP_ASSIGNMENT_DIR):
    if _dir not in sys.path:
        sys.path.append(_dir)

print('Paths added:', PACKAGE_PARENT_DIR, NLP_ASSIGNMENT_DIR)


Paths added: /content/gdrive/MyDrive/Colab Notebooks/NLP_Test /content/gdrive/MyDrive/NLP_assignment


In [3]:
# Install all dependencies needed across all three pipelines
!pip install -q transformers accelerate bitsandbytes sentencepiece sympy wikipedia-api
!pip install -q torch --index-url https://download.pytorch.org/whl/cu118
!pip install -q python-terrier sentence-transformers scikit-learn
!pip install -q protobuf latex2sympy2 faiss-cpu trafilatura requests beautifulsoup4


In [4]:
from millionaire_client import MillionaireClient, AuthenticationError
import time, json, re, random, gc
from datetime import datetime, timezone, timedelta
from pathlib import Path
from urllib.error import HTTPError
from urllib.parse import quote, urlencode
from urllib.request import Request, urlopen
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch
print('Imports complete')


Imports complete


## 1. Login & Explore the Game

In [5]:
# Credentials(store in Colab Secrets as 'poli-millionaire')
from google.colab import userdata

API_URL = "http://131.175.15.22:51111/"
USERNAME = "gary"
PASSWORD = "13790229"

client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f"Welcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed, it has: {e}")

Welcome, gary! (Role: student)


In [7]:
# List all competitions
print("=== Available Competitions ===")
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"  [{comp.id}] {comp.name} | {comp.max_levels} questions | {comp.description}")

=== Available Competitions ===
  [0] Entertainment | 15 questions | Music, Movies, Celebrities and more
  [1] Ancient History and Politics | 15 questions | The Roman Empire, The Greeks, and more
  [2] Science and Nature | 15 questions | Chemistry, Biology, Physics and similar subjects
  [3] Maths | 15 questions | Mathematics and Statistics from High School and College
  [4] Philosophy and Psychology | 15 questions | Great thinkers and the human psyche
  [5] News | 15 questions | Staying current with global breaking news


## 2. Shared Base Model â€” Qwen 2.5-7B-Instruct

All three pipelines use the same model. Load it once here; every section below
references `model` / `tokenizer` defined in this cell.


In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
ANSWER_MODEL_ID = MODEL_ID
ANSWER_MODEL_FOR_GAME = MODEL_ID

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
).eval()

answer_tokenizer = tokenizer
answer_model = model
_ANSWER_MODEL_CACHE = {MODEL_ID: (tokenizer, model)}


def load_answer_model(model_name: str = MODEL_ID):
    """Return the already-loaded shared answer model used by every pipeline."""
    requested_model = model_name or MODEL_ID
    shared_model_id = globals().get("MODEL_ID", MODEL_ID)

    if requested_model != shared_model_id:
        raise ValueError(
            f"Requested {requested_model}, but the shared loaded model is {shared_model_id}. "
            "Change MODEL_ID in the shared loader cell and rerun the notebook instead of loading a second model."
        )

    _ANSWER_MODEL_CACHE[shared_model_id] = (tokenizer, model)
    return tokenizer, model


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Model loaded once: {MODEL_ID}")
if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model loaded once: Qwen/Qwen2.5-7B-Instruct
GPU memory allocated: 5.55 GB


---
## 3. Entertainment / Science / Psychology Pipeline
_Competition IDs: 0 (Entertainment), 2 (Science & Nature), 4 (Philosophy & Psychology)_

Techniques: zero-shot, few-shot, chain-of-thought, Wikipedia RAG,
DuckDuckGo hybrid RAG, multi-model ensemble.


### 3a. Zero-Shot & Prompt Variants

In [9]:
import re
import time
import torch

#zero-shot prompt
def build_zero_shot_prompt(question_text, options):
    # Format options as A) B) C) D)
    opts = "\n".join([f"{chr(65+i)}) {o.text}" for i, o in enumerate(options)])
    return (
        f"Answer the following multiple choice question. "
        f"Reply with only the letter A, B, C, or D.\n\n"
        f"Question: {question_text}\n{opts}\n\nAnswer:"
    )

def extract_letter(text):
    # 1. Clean the text
    text = text.strip()

    # 2. Look for explicit patterns like "Answer: B" or "Final Answer: [B]" near the end
    match = re.search(r"(?:FINAL ANSWER|ANSWER|OPTION):\s*([A-D])", text.upper())
    if match:
        return match.group(1)

    # 3. Fallback: Find all isolated capital letters A, B, C, D and take the LAST one
    letters = re.findall(r"\b([A-D])\b", text.upper())
    if letters:
        return letters[-1]  # Takes the final decision made by the model

    return "A"  # Default fallback guess

def answer_with_model(question, prompt_fn=build_zero_shot_prompt, max_new_tokens=128):
    # Time the response
    t0 = time.time()

    # 1. Generate your standard question string
    raw_prompt = prompt_fn(question.text, question.options)

    # 2. Format it into the model's chat structure
    messages = [{"role": "user", "content": raw_prompt}]
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # 3. Tokenize the formatted prompt
    inputs = tokenizer(formatted_prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

    # 4. Generate the answer
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            do_sample=False,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id
        )

    # 5. CRITICAL FIX: Only extract tokens generated AFTER the prompt sequence length
    prompt_length = inputs.input_ids.shape[1]
    new_generated_tokens = outputs[0][prompt_length:]

    # Decode ONLY the new answer text
    response = tokenizer.decode(new_generated_tokens, skip_special_tokens=True)

    # Extract the choice letter from the clean text response
    letter = extract_letter(response)
    elapsed = time.time() - t0

    # Map letter to option ID (with safety lower boundary check)
    idx = ord(letter) - ord("A")
    idx = min(max(0, idx), len(question.options) - 1)

    return question.options[idx].id, letter, elapsed, response

print("Answer function defined.")

Answer function defined.


### 3b. Generic Game Loop

In [10]:
# The game loop
def play_full_game(competition_id, answer_fn, label="Model"):
    """
    Play a complete game and return results log.
    answer_fn: callable(question) -> (option_id, letter, elapsed, raw_response)
    """
    # Start the game
    game = client.game.start(competition_id=competition_id)
    print(f"\n=== Game Started: {label} | Competition {competition_id} | Session {game.session_id} ===")

    log = []

    while game.in_progress:
        q = game.current_question
        if not q:
            print("No question available, there is. Ending, the game is.")
            break

        time_left = game.time_remaining
        print(f"\n--- Level {game.current_level} | Time left: {time_left:.1f}s ---")
        print(f"Q: {q.text}")
        for opt in q.options:
            print(f"   [{opt.id}] {opt.text}")

        # Get answer from the model
        try:
            option_id, letter, elapsed, raw = answer_fn(q)
        except Exception as e:
            print(f"Model error, there is: {e}. Random answer, choosing we are.")
            option_id = random.choice(q.options).id
            letter, elapsed, raw = "?", 0.0, str(e)

        print(f"   â†’ Chose: {letter} (in {elapsed:.2f}s)")

        # Submit answer
        result = game.answer(option_id)

        entry = {
            "level": game.current_level if not result.game_over else game.current_level,
            "question": q.text,
            "correct": result.correct,
            "timed_out": result.timed_out,
            "elapsed": elapsed,
            "chosen_letter": letter,
            "earned": result.earned_amount,
            "model_raw": raw,
        }
        log.append(entry)

        if result.timed_out:
            print("   â° TIMED OUT!")
            break
        elif result.correct:
            print(f"   âœ… CORRECT! Earned: ${result.earned_amount:,.0f}")
            if result.game_over:
                print("   ðŸ† GAME COMPLETE!")
                break
        else:
            print(f"   âŒ WRONG! Final earnings: ${result.earned_amount:,.0f}")
            break

    print(f"\n=== Game Over | Reached Level: {game.current_level} | Earnings: ${game.earned_amount:,.0f} ===")
    return log, game.current_level, game.earned_amount

### 3c. Run Baseline (Zero-Shot)

In [11]:
# Competition IDs: 0=Entertainment, 1=Ancient History & Politics, 2=Science & Nature, 3=Maths, 4=Philosophy & Psychology, 5=News
COMP_ID = 0

baseline_log, baseline_level, baseline_earned = play_full_game(
    competition_id=COMP_ID,
    answer_fn=lambda q: answer_with_model(q, build_zero_shot_prompt),
    label="QWEN Zero-Shot"
)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



=== Game Started: QWEN Zero-Shot | Competition 0 | Session 336783 ===

--- Level 1 | Time left: 30.0s ---
Q: Which of Streep's early roles had the most significant impact on her career?
   [0] Julia
   [1] The Family Upstairs
   [2] Miss Julie
   [3] Alice at the Palace
   â†’ Chose: A (in 1.89s)
   âŒ WRONG! Final earnings: $0

=== Game Over | Reached Level: 1 | Earnings: $0 ===


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


### 3d. Few-Shot & Chain-of-Thought Prompt Variants

In [12]:
# Entertainment specialized few-shot data
FEW_SHOT_EXAMPLES = [
    {
        "question": "Which movie won the Academy Award for Best Picture in 2020?",
        "options": ["A) 1917", "B) Parasite", "C) Joker", "D) Once Upon a Time in Hollywood"],
        "answer": "B"
    },
    {
        "question": "Who is widely recognized as the 'King of Pop'?",
        "options": ["A) Elvis Presley", "B) Prince", "C) Michael Jackson", "D) Madonna"],
        "answer": "C"
    }
]

def build_few_shot_prompt(question_text, options):
    shots = ""
    for ex in FEW_SHOT_EXAMPLES:
        shots += f"Question: {ex['question']}\n" + "\n".join(ex['options']) + f"\nAnswer: {ex['answer']}\n\n"
    opts = "\n".join([f"{chr(65+i)}) {o.text}" for i, o in enumerate(options)])
    return (
        f"Answer multiple choice questions with only a single letter A, B, C, or D.\n\n"
        f"{shots}"
        f"Question: {question_text}\n{opts}\nAnswer:"
    )

def build_cot_prompt(question_text, options):
    opts = "\n".join([f"{chr(65+i)}) {o.text}" for i, o in enumerate(options)])
    return (
        f"Answer the following question. Think briefly, then give your final answer as a single letter.\n\n"
        f"Question: {question_text}\n{opts}\n\n"
        f"Reasoning: Let me think step by step.\nFinal Answer:"
    )

print("Prompt variants successfully adjusted for Entertainment trivia parsing.")

Prompt variants successfully adjusted for Entertainment trivia parsing.


In [13]:
# Compare prompts offline on a sample â€” NOT via live game (save API calls)
# Manually define a test question to compare prompt styles
sample_text = "Which planet is known as the Red Planet?"

class FakeOption:
    def __init__(self, id_, text):
        self.id = id_
        self.text = text

sample_opts = [FakeOption(1,"Mars"), FakeOption(2,"Venus"), FakeOption(3,"Jupiter"), FakeOption(4,"Saturn")]

class FakeQ:
    def __init__(self):
        self.text = sample_text
        self.options = sample_opts

fq = FakeQ()

print("=== Zero-Shot ===")
print(build_zero_shot_prompt(fq.text, fq.options))
print("\n=== Few-Shot ===")
print(build_few_shot_prompt(fq.text, fq.options))
print("\n=== Chain-of-Thought ===")
print(build_cot_prompt(fq.text, fq.options))

=== Zero-Shot ===
Answer the following multiple choice question. Reply with only the letter A, B, C, or D.

Question: Which planet is known as the Red Planet?
A) Mars
B) Venus
C) Jupiter
D) Saturn

Answer:

=== Few-Shot ===
Answer multiple choice questions with only a single letter A, B, C, or D.

Question: Which movie won the Academy Award for Best Picture in 2020?
A) 1917
B) Parasite
C) Joker
D) Once Upon a Time in Hollywood
Answer: B

Question: Who is widely recognized as the 'King of Pop'?
A) Elvis Presley
B) Prince
C) Michael Jackson
D) Madonna
Answer: C

Question: Which planet is known as the Red Planet?
A) Mars
B) Venus
C) Jupiter
D) Saturn
Answer:

=== Chain-of-Thought ===
Answer the following question. Think briefly, then give your final answer as a single letter.

Question: Which planet is known as the Red Planet?
A) Mars
B) Venus
C) Jupiter
D) Saturn

Reasoning: Let me think step by step.
Final Answer:


In [14]:
# Run few-shot game â€” compare with baseline result above
fewshot_log, fewshot_level, fewshot_earned = play_full_game(
    competition_id=COMP_ID,
    answer_fn=lambda q: answer_with_model(q, build_few_shot_prompt),
    label="Few-Shot"
)


=== Game Started: Few-Shot | Competition 0 | Session 336788 ===

--- Level 1 | Time left: 30.0s ---
Q: What is the fundamental principle of Drake's approach to songwriting?
   [0] Writing exclusively about his personal relationships
   [1] Focusing on traditional rap battle themes
   [2] Blending different genres to create unique soundscapes
   [3] Incorporating unexpected collaborations with other artists
   â†’ Chose: C (in 0.60s)
   âœ… CORRECT! Earned: $100

--- Level 2 | Time left: 30.0s ---
Q: What is the fundamental principle of Casablanca's plot?
   [0] A Czechoslovak resistance leader's escape from Vichy-controlled Casablanca
   [1] An American expatriate's struggle against the Nazis
   [2] A romantic love story between Humphrey Bogart and Ingrid Bergman
   [3] A complex moral dilemma involving love, loyalty, and resistance
   â†’ Chose: D (in 0.52s)
   âœ… CORRECT! Earned: $200

--- Level 3 | Time left: 30.0s ---
Q: What term describes David Bowie's reinvention and visual pr

### 3e. RAG â€” Wikipedia (Entertainment / Science)

Uses `search_wikipedia_deep` + an entertainment-aware RAG prompt.

In [15]:
import requests
import re
import time

def search_wikipedia_deep(query, max_chars=1200):
    """
    Deeper Wikipedia extract grabber to catch tracklists, cast lists,
    and detailed table indexes missing from short summaries.
    """
    clean = re.sub(r'[^\w\s]', '', query)[:60].strip()

    # Phase 1: Search API to get the correct matching title
    search_url = "https://en.wikipedia.org/w/api.php"
    search_params = {
        "action": "query", "list": "search",
        "srsearch": clean, "format": "json", "srlimit": 1
    }
    try:
        r = requests.get(search_url, params=search_params, timeout=5).json()
        results = r.get("query", {}).get("search", [])
        if not results:
            return ""
        title = results[0]["title"]

        # Phase 2: Request full un-summarized text section extract
        content_params = {
            "action": "query", "prop": "extracts",
            "explaintext": 1, "titles": title, "format": "json", "exintro": 0
        }
        resp = requests.get(search_url, params=content_params, timeout=5).json()
        pages = resp["query"]["pages"]
        page_id = list(pages.keys())[0]

        extract = pages[page_id].get("extract", "")
        return extract[:max_chars]
    except Exception:
        return ""

def extract_entertainment_query(question_text, options):
    """
    Concatenates target choices to the question search query
    so negative constraint tracking works properly.
    """
    # Isolate key elements like text in quotes (e.g. "Born to Die")
    quoted_terms = re.findall(r'"([^"]*)"', question_text)
    options_string = " ".join([o.text for o in options])

    # Strip common filler stop phrases
    clean_q = re.sub(r'(Which of these|is not|featured on|the standard version of|the following|correct answer)', '', question_text, flags=re.IGNORECASE)

    if quoted_terms:
        return f'"{quoted_terms[0]}" {options_string}'[:90]
    return f"{clean_q.strip()} {options_string}"[:90]

def build_entertainment_rag_prompt(question_text, options, context=""):
    """
    Advanced prompt directing process of elimination for negative properties (NOT, EXCEPT).
    """
    opts = "\n".join([f"{chr(65+i)}) {o.text}" for i, o in enumerate(options)])
    ctx_block = f"Background Context Information:\n{context}\n\n" if context else ""

    return (
        f"{ctx_block}"
        f"Task: Solve this multiple choice entertainment trivia question based ONLY on the text above.\n"
        f"CRITICAL RULES:\n"
        f"1. If the question contains words like 'NOT', 'FALSE', or 'EXCEPT', use process of elimination. Eliminate any options explicitly verified by the context text and pick the one outlier that remains.\n"
        f"2. Output strictly a single capital letter matching the answer (A, B, C, or D).\n\n"
        f"Question: {question_text}\n"
        f"Options:\n{opts}\n\n"
        f"Final Answer (Letter Only):"
    )

def answer_with_rag(question):
    # Pass options array to feed query synthesis loop
    query = extract_entertainment_query(question.text, question.options)
    context = search_wikipedia_deep(query)

    if context:
        print(f"   [RAG Active] Retrieved context data frame for search parameters.")
    else:
        print("   [RAG Warning] Flying blind without structural text records.")

    return answer_with_model(
        question,
        lambda q_text, opts: build_entertainment_rag_prompt(q_text, opts, context),
        max_new_tokens=16 # Kept short to prevent model from writing chat justifications
    )

### 3f. RAG â€” Wikipedia with News Query Extractor

Shares `search_wikipedia_deep`; swaps in a news-oriented query builder and prompt.
Useful for competition 5 warm-up runs against Wikipedia.


In [16]:
def build_news_rag_prompt(question_text, options, context=""):
    """
    Optimized for News and Events: Forces the model to carefully inspect
    historical timelines, official roles, and factual details from the Wikipedia context.
    """
    opts = "\n".join([f"{chr(65+i)}) {o.text}" for i, o in enumerate(options)])
    ctx_block = f"Background Context Information (Wikipedia Extract):\n{context}\n\n" if context else ""

    return (
        f"{ctx_block}"
        f"Task: Solve this current events and news multiple choice question using the background text above.\n"
        f"CRITICAL RULES:\n"
        f"1. Pay close attention to exact dates, years, official political titles, and specific country/geographic actions mentioned in the text.\n"
        f"2. Output strictly a single capital letter matching the correct choice (A, B, C, or D).\n\n"
        f"Question: {question_text}\n"
        f"Options:\n{opts}\n\n"
        f"Final Answer (Letter Only):"
    )

def answer_with_rag(question):
    # Call the new news query extractor instead of entertainment
    query = extract_news_query(question.text, question.options)
    print(f"   [Wikipedia News Search] Query: '{query}'")

    # Keep using your deep lookup engine exactly as it was
    context = search_wikipedia_deep(query)

    if context:
        print(f"   [RAG Active] Retrieved context data frame from Wikipedia.")
    else:
        print("   [RAG Warning] Flying blind without structural text records.")

    return answer_with_model(
        question,
        lambda q_text, opts: build_news_rag_prompt(q_text, opts, context),
        max_new_tokens=16
    )

### 3g. Run Wikipedia RAG Game

In [17]:
# Run RAG game
rag_log, rag_level, rag_earned = play_full_game(
    competition_id=0,
    answer_fn=answer_with_rag,
    label="Model + Wikipedia RAG"
)


=== Game Started: Model + Wikipedia RAG | Competition 0 | Session 336793 ===

--- Level 1 | Time left: 30.0s ---
Q: How does Harvey Dent's character transformation relate to the film's theme of moral corruption?
   [0] His transformation is unrelated to the film's theme
   [1] His transformation illustrates that corruption is a result of external forces only
   [2] His transformation shows that even heroes can be corrupted by power
   [3] His transformation proves that corruption is easily avoidable
Model error, there is: name 'extract_news_query' is not defined. Random answer, choosing we are.
   â†’ Chose: ? (in 0.00s)
   âŒ WRONG! Final earnings: $0

=== Game Over | Reached Level: 1 | Earnings: $0 ===


### 3h. Hybrid RAG â€” DuckDuckGo + Wikipedia

Tries DuckDuckGo first, falls back to Wikipedia. Best for pop-culture/entertainment.


In [18]:
# Multi-Source RAG Setup â€” DuckDuckGo Live Search + Wikipedia Fallback
import requests
import re

def search_duckduckgo(query, max_chars=600):
    """
    Queries DuckDuckGo's free API for an instant abstract summary.
    Perfect for pop-culture entities, famous tracks, actors, and media questions.
    """
    clean_query = query.strip()
    url = f"https://api.duckduckgo.com/?q={requests.utils.quote(clean_query)}&format=json&no_html=1"
    try:
        response = requests.get(url, timeout=4)
        if response.status_code == 200:
            data = response.json()
            # DuckDuckGo provides 'AbstractText' for broad definitions
            abstract = data.get("AbstractText", "")
            if abstract:
                return abstract[:max_chars]

            # Alternative fallback: look inside RelatedTopics list snippets
            related = data.get("RelatedTopics", [])
            if related and "Text" in related[0]:
                return related[0]["Text"][:max_chars]
    except Exception:
        pass
    return ""

def search_wikipedia(query, max_chars=600):
    """Search Wikipedia and return a summary snippet as a safety fallback."""
    clean = re.sub(r'[^\w\s]', '', query)[:60]
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{clean.replace(' ', '_')}"
    try:
        r = requests.get(url, timeout=4)
        if r.status_code == 200:
            extract = r.json().get("extract", "")
            if extract:
                return extract[:max_chars]
    except Exception:
        pass

    # Secondary deep title search fallback
    try:
        params = {
            "action": "query", "list": "search",
            "srsearch": query, "format": "json", "srlimit": 1
        }
        r = requests.get("https://en.wikipedia.org/w/api.php", params=params, timeout=4)
        results = r.json().get("query", {}).get("search", [])
        if results:
            title = results[0]["title"]
            return search_wikipedia(title, max_chars)
    except Exception:
        pass

    return ""

def extract_key_terms_with_options(question_text, options):
    """
    Combines question entities with candidate choices.
    This guarantees that the search checks for the tracks/choices explicitly.
    """
    # Isolate any quoted strings first (e.g. "Born to Die")
    quoted = re.findall(r'"([^"]*)"', question_text)
    options_str = " ".join([o.text for o in options])

    # Strip basic filler text
    clean_q = re.sub(r'(Which of these|is not|featured on|the standard version of|the following|correct answer)', '', question_text, flags=re.IGNORECASE)

    if quoted:
        return f'"{quoted[0]}" {options_str}'[:90]
    return f"{clean_q.strip()} {options_str}"[:90]

def build_hybrid_rag_prompt(question_text, options, context=""):
    """
    A smart prompt directing process of elimination when a context string is found.
    """
    opts = "\n".join([f"{chr(65+i)}) {o.text}" for i, o in enumerate(options)])
    ctx_block = f"Background Context Document:\n{context}\n\n" if context else ""
    return (
        f"{ctx_block}"
        f"Task: Answer the multiple-choice trivia question based on the background context provided above.\n"
        f"Rule: If the question contains words like 'NOT', 'EXCEPT', or 'FALSE', eliminate options matched by the context and pick the outlier.\n\n"
        f"Question: {question_text}\n"
        f"Options:\n{opts}\n\n"
        f"Final Answer (Letter Only):"
    )

def answer_with_rag(question):
    """
    Ensemble Retrieval Engine: Queries DuckDuckGo first,
    then falls back to Wikipedia if no result is returned.
    """
    query = extract_key_terms_with_options(question.text, question.options)

    # Step 1: Try Live Web Summary via DuckDuckGo
    context = search_duckduckgo(query)
    source_used = "DuckDuckGo"

    # Step 2: Fall back to Wikipedia if DuckDuckGo came up empty
    if not context:
        context = search_wikipedia(query)
        source_used = "Wikipedia"

    if context:
        print(f"   [RAG Active] Context fetched via {source_used}: '{context[:70]}...'")
    else:
        print("   [RAG Warning] Both sources empty! Flying blind via zero-shot memory.")

    return answer_with_model(
        question,
        lambda q_text, opts: build_hybrid_rag_prompt(q_text, opts, context),
        max_new_tokens=16
    )

print("Hybrid RAG engine successfully compiled! DuckDuckGo + Wikipedia are active.")

Hybrid RAG engine successfully compiled! DuckDuckGo + Wikipedia are active.


In [19]:
# Run the Multi-Source Hybrid RAG on the Entertainment competition
rag_log, rag_level, rag_earned = play_full_game(
    competition_id=0,
    answer_fn=answer_with_rag,
    label="Qwen2.5 + Hybrid RAG (DDG + Wiki)"
)


=== Game Started: Qwen2.5 + Hybrid RAG (DDG + Wiki) | Competition 0 | Session 336795 ===

--- Level 1 | Time left: 30.0s ---
Q: Which of the following best describes the connection between Drake's personal life and his music?
   [0] His music avoids mentioning personal life
   [1] His music focuses solely on fictional stories
   [2] His music often reflects personal experiences and emotions
   [3] His music is primarily about political commentary
   [RAG Warning] Both sources empty! Flying blind via zero-shot memory.
   â†’ Chose: C (in 0.64s)
   âœ… CORRECT! Earned: $100

--- Level 2 | Time left: 30.0s ---
Q: How does Quentin Tarantino's use of dialogue in his films differ from traditional dialogue in other films?
   [0] It is more formal and polite
   [1] It is less frequent and more sparse
   [2] It often includes mundane conversations with popular culture references
   [3] It is written in a foreign language
   [RAG Warning] Both sources empty! Flying blind via zero-shot memory.
 

### 3i. Multi-Model Ensemble (Majority Vote)

In [20]:
# Ensemble: majority vote across models
def answer_ensemble(question):
    votes = {}
    details = []

    models_to_use = [
        ("ZeroShot", lambda q: answer_with_model(q, build_zero_shot_prompt)),
        ("FewShot",  lambda q: answer_with_model(q, build_few_shot_prompt)),
        ("CoT",      lambda q: answer_with_model(q, build_cot_prompt)),
    ]

    for name, fn in models_to_use:
        try:
            opt_id, letter, elapsed, raw = fn(question)
            votes[letter] = votes.get(letter, 0) + 1
            details.append((name, letter, opt_id, elapsed))
            print(f"   [{name}] voted: {letter}")
        except Exception as e:
            print(f"   [{name}] failed: {e}")

    if not votes:
        # All models failed, we choose random answer
        chosen = random.choice(question.options)
        return chosen.id, "?", 0.0, "All models failed"

    # Find most voted letter
    best_letter = max(votes, key=votes.get)
    print(f"   [ENSEMBLE] Majority vote â†’ {best_letter} ({votes[best_letter]}/{len(models_to_use)} votes)")

    # Find option ID for the winning letter
    idx = min(ord(best_letter) - ord("A"), len(question.options) - 1)
    chosen_id = question.options[idx].id
    avg_elapsed = sum(d[3] for d in details) / len(details)

    return chosen_id, best_letter, avg_elapsed, str(votes)

print("Ensemble function ready")

Ensemble function ready


In [21]:
# Run ensemble game, we shall
ensemble_log, ensemble_level, ensemble_earned = play_full_game(
    competition_id=COMP_ID,
    answer_fn=answer_ensemble,
    label="Multi-Model Ensemble"
)


=== Game Started: Multi-Model Ensemble | Competition 0 | Session 336797 ===

--- Level 1 | Time left: 30.0s ---
Q: What is the fundamental principle of the film 'Downfall'?
   [0] To provide a comedic view of Hitler's final days
   [1] To make a historically accurate account of the last days of Nazi Germany
   [2] To focus solely on the military strategies of World War II
   [3] To glorify the actions of Adolf Hitler
   [ZeroShot] voted: B
   [FewShot] voted: B
   [CoT] voted: B
   [ENSEMBLE] Majority vote â†’ B (3/3 votes)
   â†’ Chose: B (in 0.51s)
   âœ… CORRECT! Earned: $100

--- Level 2 | Time left: 30.0s ---
Q: Which of the following best describes Elvis Presley's vocal technique during his early career?
   [0] Raw and emotive
   [1] Hushed and subtle
   [2] Smooth and sophisticated
   [3] Clear and melodic
   [ZeroShot] voted: A
   [FewShot] voted: A
   [CoT] voted: A
   [ENSEMBLE] Majority vote â†’ A (3/3 votes)
   â†’ Chose: A (in 0.48s)
   âœ… CORRECT! Earned: $200

--- Leve

---
## 4. History Pipeline
_Competition ID: 1 (Ancient History & Politics)_

Techniques: PyTerrier BM25 + sentence-embedding reranker + optional cross-encoder,
direct logit scoring with shuffled option orders, agentic tool router.


In [22]:
comp_id = 1  # Ancient History & Politics


### 4a. Wikipedia Retrieval Helpers

In [23]:
import json
import re
from urllib.parse import quote, urlencode
from urllib.error import HTTPError
from urllib.request import Request, urlopen
import time

WIKIPEDIA_API = "https://en.wikipedia.org/w/api.php"
WIKIPEDIA_USER_AGENT = "PoliMillionaireNLP/1.0 student project"
WIKIPEDIA_REQUEST_DELAY_SECONDS = 0.8
WIKIPEDIA_429_BACKOFF_SECONDS = 4.0
WIKIPEDIA_MAX_RETRIES = 2
MAX_WIKIPEDIA_SEARCH_QUERIES = 2
_LAST_WIKIPEDIA_REQUEST_TIME = 0.0
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "by", "for", "from",
    "has", "have", "how", "in", "is", "it", "its", "of", "on", "or", "that",
    "the", "their", "there", "these", "this", "those", "to", "was", "were",
    "what", "when", "where", "which", "who", "why", "with", "according", "article",
    "considered", "important", "goal", "goals", "main", "primary", "following"
}


def question_to_text(question) -> str:
    """Accept a string, a dict, or a millionaire_client Question object."""
    if hasattr(question, "text"):
        return str(question.text)
    if isinstance(question, dict) and "text" in question:
        return str(question["text"])
    return str(question)


def normalize_wikipedia_text(text: str) -> str:
    """Clean a plain Wikipedia extract enough for later NLP steps."""
    text = str(text).replace("\xa0", " ")
    text = re.sub(r"\[\d+\]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def tokenize(text: str) -> list[str]:
    return [token for token in re.findall(r"[a-z0-9]+", str(text).lower()) if len(token) > 1]


def expand_term(token: str) -> set[str]:
    """Tiny synonym/variant helper for common historical wording traps."""
    variants = {token}
    if token == "roman":
        variants.update({"rome", "romans"})
    elif token in {"rome", "romans"}:
        variants.add("roman")
    return variants


def extract_keywords(text: str, limit: int = 10) -> list[str]:
    keywords = []
    seen = set()
    for token in tokenize(text):
        if token in STOPWORDS or token in seen:
            continue
        keywords.append(token)
        seen.add(token)
        if len(keywords) >= limit:
            break
    return keywords


def capital_context_phrases(question_text: str) -> list[str]:
    """Build focused phrases like 'Roman marriage' from capitalized topic words."""
    words = re.findall(r"[A-Za-z][A-Za-z'-]*", question_text)
    phrases = []
    for index, word in enumerate(words):
        if not word[:1].isupper() or word.lower() in STOPWORDS:
            continue
        phrase_words = [word]
        for next_word in words[index + 1:index + 4]:
            if next_word.lower() in STOPWORDS:
                break
            phrase_words.append(next_word)
        if len(phrase_words) > 1:
            phrases.append(" ".join(phrase_words))
    return phrases


def raw_option_text(option) -> str:
    """Read option text without depending on later notebook cells."""
    if hasattr(option, "text"):
        return str(option.text)
    if isinstance(option, dict):
        return str(option.get("text", ""))
    return str(option)


def question_options(question, options=None) -> list:
    """Return answer options from an explicit argument or a Question object."""
    if options is not None:
        return list(options)
    if hasattr(question, "options"):
        return list(question.options)
    if isinstance(question, dict) and "options" in question:
        return list(question["options"])
    return []


def dedupe_queries(queries: list[str]) -> list[str]:
    deduped = []
    seen = set()
    for query in queries:
        normalized = normalize_wikipedia_text(query).lower()
        if normalized and normalized not in seen:
            deduped.append(query)
            seen.add(normalized)
    return deduped


def build_base_wikipedia_search_queries(question) -> list[str]:
    """Create focused question-only Wikipedia queries."""
    question_text = question_to_text(question)
    cleaned = re.sub(r"\baccording to (?:the )?(?:article|text|passage)\b", " ", question_text, flags=re.IGNORECASE)
    cleaned = normalize_wikipedia_text(cleaned)
    keywords = extract_keywords(cleaned, limit=10)

    capital_words = []
    for word in re.findall(r"[A-Za-z][A-Za-z'-]*", cleaned):
        normalized_word = word.strip("'-")
        lowered = normalized_word.lower()
        if normalized_word[:1].isupper() and lowered not in STOPWORDS and len(lowered) > 2:
            capital_words.append(normalized_word)

    queries = []
    if len(capital_words) >= 2:
        queries.append(" ".join(capital_words[:4]))
    queries.extend(capital_context_phrases(cleaned))
    if keywords:
        queries.append(" ".join(keywords[:6]))
    if len(keywords) >= 2:
        queries.append(" ".join(keywords[:2]))
    queries.append(cleaned)
    queries.append(question_text)
    return dedupe_queries(queries)


def build_option_wikipedia_search_queries(question, options=None) -> list[str]:
    """Create one concise search query per option, balanced across all choices."""
    question_text = question_to_text(question)
    cleaned = re.sub(r"\baccording to (?:the )?(?:article|text|passage)\b", " ", question_text, flags=re.IGNORECASE)
    cleaned = normalize_wikipedia_text(cleaned)
    question_keywords = extract_keywords(cleaned, limit=6)
    queries = []
    for option in question_options(question, options):
        option_keywords = extract_keywords(raw_option_text(option), limit=6)
        if option_keywords:
            queries.append(" ".join((question_keywords[:4] + option_keywords[:4])[:8]))
    return dedupe_queries(queries)




def wikipedia_request(params: dict, timeout: float = 6.0, deadline_monotonic=None) -> dict:
    global _LAST_WIKIPEDIA_REQUEST_TIME

    url = f"{WIKIPEDIA_API}?{urlencode(params)}"
    request = Request(url, headers={"User-Agent": WIKIPEDIA_USER_AGENT})

    for attempt in range(WIKIPEDIA_MAX_RETRIES + 1):
        if deadline_monotonic is not None and time.monotonic() >= deadline_monotonic:
            raise TimeoutError("Wikipedia request skipped because the question deadline was reached")

        elapsed_since_last = time.monotonic() - _LAST_WIKIPEDIA_REQUEST_TIME
        sleep_for = WIKIPEDIA_REQUEST_DELAY_SECONDS - elapsed_since_last
        if sleep_for > 0:
            if deadline_monotonic is not None:
                remaining = deadline_monotonic - time.monotonic()
                if remaining <= 0:
                    raise TimeoutError("Wikipedia delay skipped because the question deadline was reached")
                sleep_for = min(sleep_for, remaining)
            time.sleep(sleep_for)

        request_timeout = timeout
        if deadline_monotonic is not None:
            remaining = deadline_monotonic - time.monotonic()
            if remaining <= 0:
                raise TimeoutError("Wikipedia request skipped because the question deadline was reached")
            request_timeout = min(timeout, max(0.25, remaining))

        try:
            with urlopen(request, timeout=request_timeout) as response:
                _LAST_WIKIPEDIA_REQUEST_TIME = time.monotonic()  # ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â¦ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã¢â‚¬Å“ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¦ only on success
                data = json.loads(response.read().decode("utf-8"))
                return data

        except HTTPError as exc:
            if exc.code == 429:
                _LAST_WIKIPEDIA_REQUEST_TIME = time.monotonic() - WIKIPEDIA_REQUEST_DELAY_SECONDS
                raise TimeoutError("Wikipedia rate limited; skipping live retry inside timed game")
            if attempt >= WIKIPEDIA_MAX_RETRIES:
                _LAST_WIKIPEDIA_REQUEST_TIME = time.monotonic() - WIKIPEDIA_REQUEST_DELAY_SECONDS
                raise

            retry_after = exc.headers.get("Retry-After")
            try:
                wait_seconds = float(retry_after) if retry_after else WIKIPEDIA_429_BACKOFF_SECONDS * (attempt + 1)
            except ValueError:
                wait_seconds = WIKIPEDIA_429_BACKOFF_SECONDS * (attempt + 1)

            if deadline_monotonic is not None:
                remaining = deadline_monotonic - time.monotonic()
                if remaining <= 0 or wait_seconds >= remaining:
                    _LAST_WIKIPEDIA_REQUEST_TIME = time.monotonic() - WIKIPEDIA_REQUEST_DELAY_SECONDS
                    raise TimeoutError("Wikipedia rate-limit backoff would exceed the question deadline")
                wait_seconds = min(wait_seconds, remaining)

            print(f"Wikipedia rate limit hit. Waiting {wait_seconds:.1f}s before retry {attempt + 1}/{WIKIPEDIA_MAX_RETRIES}...")
            time.sleep(wait_seconds)
            _LAST_WIKIPEDIA_REQUEST_TIME = time.monotonic()  # reset AFTER the backoff sleep



def search_wikipedia(query: str, limit: int = 5, timeout: float = 6.0, deadline_monotonic=None) -> list[dict]:
    """Search Wikipedia and return candidate pages for one query string."""
    query = normalize_wikipedia_text(query)
    data = wikipedia_request(
        {
            "action": "query",
            "list": "search",
            "srsearch": query,
            "srlimit": limit,
            "format": "json",
            "utf8": 1,
            "redirects": 1,
        },
        timeout=timeout,
        deadline_monotonic=deadline_monotonic,
    )

    results = data.get("query", {}).get("search", [])
    formatted_results = [
        {
            "title": item.get("title", ""),
            "page_id": item.get("pageid"),
            "snippet": normalize_wikipedia_text(re.sub(r"<[^>]+>", " ", item.get("snippet", ""))),
            "query": query,
            "search_rank": rank,
        }
        for rank, item in enumerate(results, start=1)
    ]
    return formatted_results


def collect_wikipedia_candidates(question, per_query_limit: int = 5, timeout: float = 6.0, max_search_queries=None, options=None, deadline_monotonic=None) -> list[dict]:
    """Search capped focused queries and deduplicate candidate pages by title."""
    candidates_by_title = {}
    base_queries = build_base_wikipedia_search_queries(question)
    option_queries = build_option_wikipedia_search_queries(question, options=options)
    if max_search_queries is None:
        max_search_queries = MAX_WIKIPEDIA_SEARCH_QUERIES
    if max_search_queries is not None:
        base_budget = max(1, int(max_search_queries * 0.6))
        option_budget = max(0, max_search_queries - base_budget)
        search_queries = dedupe_queries(base_queries[:base_budget] + option_queries[:option_budget])
        if len(search_queries) < max_search_queries:
            search_queries = dedupe_queries(search_queries + base_queries + option_queries)[:max_search_queries]
    else:
        search_queries = dedupe_queries(base_queries + option_queries)
    for query in search_queries:
        if deadline_monotonic is not None and time.monotonic() >= deadline_monotonic:
            print("Wikipedia search budget exhausted; using candidates collected so far.")
            break
        try:
            results = search_wikipedia(query, limit=per_query_limit, timeout=timeout, deadline_monotonic=deadline_monotonic)
        except Exception as exc:
            print(f"Wikipedia search skipped for {query!r}: {exc}")
            continue

        for result in results:
            title_key = result["title"].lower()
            if title_key not in candidates_by_title:
                candidates_by_title[title_key] = result
            else:
                candidates_by_title[title_key]["search_rank"] = min(
                    candidates_by_title[title_key]["search_rank"],
                    result["search_rank"],
                )
    return list(candidates_by_title.values())


def core_question_terms(question) -> list[str]:
    """Terms that must anchor retrieval: quoted terms and named entities in the question."""
    question_text = question_to_text(question)
    terms = []
    for phrase in re.findall(r"['\"]([^'\"]{3,80})['\"]", question_text):
        terms.extend(tokenize(phrase))
    for word in re.findall(r"\b[A-Z][A-Za-z0-9'-]{2,}\b", question_text):
        lowered = word.lower().strip("'-")
        if lowered not in STOPWORDS:
            terms.append(lowered)
    return list(dict.fromkeys(terms))[:8]


def candidate_relevance_score(candidate: dict, question, options=None) -> float:
    """Score title/snippet overlap with question keywords; penalize very generic one-word titles."""
    question_text = question_to_text(question)
    keywords = extract_keywords(question_text, limit=10)
    option_keywords = []
    for option in question_options(question, options):
        option_keywords.extend(extract_keywords(raw_option_text(option), limit=5))
    candidate_text = f"{candidate.get('title', '')} {candidate.get('snippet', '')}"
    candidate_terms = set(tokenize(candidate_text))

    matched = 0
    for keyword in keywords:
        if expand_term(keyword) & candidate_terms:
            matched += 1

    overlap = matched / max(1, len(keywords))
    title_terms = tokenize(candidate.get("title", ""))
    core_terms = core_question_terms(question)
    core_overlap = len([term for term in core_terms if expand_term(term) & candidate_terms]) / max(1, len(core_terms)) if core_terms else 1.0
    option_overlap = len(set(option_keywords) & candidate_terms) / max(1, len(set(option_keywords))) if option_keywords else 0.0
    rank_bonus = 1.0 / max(1, candidate.get("search_rank", 1))
    generic_penalty = 0.35 if len(title_terms) == 1 and len(keywords) > 1 else 0.0
    missing_core_penalty = 0.85 if core_terms and core_overlap == 0 else 0.0
    generic_title_penalty = 0.30 if title_terms and core_terms and not (set(title_terms) & set(core_terms)) and len(set(title_terms) & set(keywords)) <= 1 else 0.0

    return (1.6 * overlap) + (0.20 * option_overlap) + (0.80 * core_overlap) + (0.25 * rank_bonus) - generic_penalty - missing_core_penalty - generic_title_penalty


def rank_wikipedia_candidates(question, per_query_limit: int = 5, timeout: float = 6.0, max_search_queries=None, options=None, deadline_monotonic=None) -> list[dict]:
    candidates = collect_wikipedia_candidates(question, per_query_limit=per_query_limit, timeout=timeout, max_search_queries=max_search_queries, options=options, deadline_monotonic=deadline_monotonic)
    for candidate in candidates:
        candidate["candidate_score"] = candidate_relevance_score(candidate, question, options=options)
    return sorted(candidates, key=lambda item: item["candidate_score"], reverse=True)


def fetch_wikipedia_extract(title: str, timeout: float = 6.0, deadline_monotonic=None) -> dict:
    """Fetch a Wikipedia page as a plain-text document."""
    data = wikipedia_request(
        {
            "action": "query",
            "prop": "extracts|info",
            "explaintext": 1,
            "exsectionformat": "plain",
            "inprop": "url",
            "titles": title,
            "format": "json",
            "utf8": 1,
            "redirects": 1,
        },
        timeout=timeout,
        deadline_monotonic=deadline_monotonic,
    )

    pages = data.get("query", {}).get("pages", {})
    page = next(iter(pages.values()), {}) if pages else {}
    document = {
        "title": page.get("title", title),
        "page_id": page.get("pageid"),
        "url": page.get("fullurl") or f"https://en.wikipedia.org/wiki/{quote(title.replace(' ', '_'))}",
        "text": normalize_wikipedia_text(page.get("extract", "")),
    }
    return document


def get_wikipedia_documents_for_question(question, top_n: int = 5, per_query_limit: int = 5, timeout: float = 6.0, max_search_queries=None, options=None, deadline_monotonic=None) -> list[dict]:
    """
    Return the top N related Wikipedia documents for a question.

    This avoids the trap of trusting only Wikipedia's first result for the full question.
    """
    query = question_to_text(question)
    candidates = rank_wikipedia_candidates(question, per_query_limit=per_query_limit, timeout=timeout, max_search_queries=max_search_queries, options=options, deadline_monotonic=deadline_monotonic)
    documents = []

    min_candidate_score = float(globals().get("MIN_WIKIPEDIA_CANDIDATE_SCORE", 0.0))
    for candidate in candidates[:top_n]:
        if candidate.get("candidate_score", 0.0) < min_candidate_score:
            print(f"Wikipedia candidate skipped for low relevance: {candidate.get('title')!r} score={candidate.get('candidate_score', 0.0):.3f}")
            continue
        if deadline_monotonic is not None and time.monotonic() >= deadline_monotonic:
            print("Wikipedia fetch budget exhausted; using documents fetched so far.")
            break
        try:
            document = fetch_wikipedia_extract(candidate["title"], timeout=timeout, deadline_monotonic=deadline_monotonic)
        except Exception as exc:
            print(f"Wikipedia page skipped for {candidate['title']!r}: {exc}")
            continue

        document["query"] = query
        document["matched_query"] = candidate.get("query")
        document["search_rank"] = candidate.get("search_rank")
        document["candidate_score"] = candidate.get("candidate_score", 0.0)
        document["snippet"] = candidate.get("snippet", "")
        document["search_results"] = candidates
        documents.append(document)

    return documents




### 4b. Chunk Retrieval & Reranking (BM25 + Sentence Embeddings + Cross-Encoder)

In [24]:
import math
import time
import torch
import pyterrier as pt
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import shutil
import tempfile


def split_sentences(text: str) -> list[str]:
    """Sentence splitter for clean Wikipedia text."""
    text = normalize_wikipedia_text(text)
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [sentence.strip() for sentence in sentences if len(sentence.strip()) >= 40]


def build_rag_chunks(documents: list[dict], sentences_per_chunk: int = 5, overlap: int = 2) -> list[dict]:
    """Split the top-N Wikipedia documents into overlapping evidence chunks."""
    chunks = []
    step = max(1, sentences_per_chunk - overlap)

    for doc_index, doc in enumerate(documents):
        sentences = split_sentences(doc.get("text", ""))
        for start in range(0, len(sentences), step):
            chunk_sentences = sentences[start:start + sentences_per_chunk]
            if not chunk_sentences:
                break
            chunk_text = " ".join(chunk_sentences)
            if len(chunk_text) < 120:
                continue
            chunks.append(
                {
                    "doc_index": doc_index,
                    "chunk_index": len(chunks),
                    "title": doc.get("title", ""),
                    "url": doc.get("url", ""),
                    "text": chunk_text,
                    "document_score": float(doc.get("candidate_score", 0.0)),
                }
            )
            if start + sentences_per_chunk >= len(sentences):
                break

    return chunks


def lexical_similarity(query: str, text: str) -> float:
    """Fallback score if sklearn is unavailable."""
    query_terms = set(extract_keywords(query, limit=20))
    text_terms = set(tokenize(text))
    if not query_terms or not text_terms:
        return 0.0
    overlap = len(query_terms & text_terms) / len(query_terms)
    return overlap


SENTENCE_EMBEDDING_MODEL_ID = globals().get(
    "SENTENCE_EMBEDDING_MODEL_ID",
    "sentence-transformers/all-MiniLM-L6-v2",
)
_SENTENCE_EMBEDDING_CACHE = globals().setdefault("_SENTENCE_EMBEDDING_CACHE", {})


def normalize_sentence_embedding_model_id(model_name: str | None = None) -> str:
    """Use one cache key for equivalent MiniLM model names across pipelines."""
    model_name = model_name or SENTENCE_EMBEDDING_MODEL_ID
    if model_name == "all-MiniLM-L6-v2":
        return SENTENCE_EMBEDDING_MODEL_ID
    return model_name


def load_sentence_embedding_model(model_name: str = SENTENCE_EMBEDDING_MODEL_ID):
    """Load a small sentence embedding model once and reuse it across pipelines."""
    model_name = normalize_sentence_embedding_model_id(model_name)
    if model_name not in _SENTENCE_EMBEDDING_CACHE:
        device = globals().get("SENTENCE_EMBEDDING_DEVICE", "cpu")
        _SENTENCE_EMBEDDING_CACHE[model_name] = SentenceTransformer(model_name, device=device)
    return _SENTENCE_EMBEDDING_CACHE[model_name]


CROSS_ENCODER_MODEL_ID = globals().get("CROSS_ENCODER_MODEL_ID", "cross-encoder/ms-marco-MiniLM-L-6-v2")
_CROSS_ENCODER_CACHE = globals().setdefault("_CROSS_ENCODER_CACHE", {})


def load_cross_encoder_model(model_name: str = CROSS_ENCODER_MODEL_ID):
    """Load an optional stronger reranker once and reuse it across pipelines."""
    if model_name not in _CROSS_ENCODER_CACHE:
        device = globals().get("CROSS_ENCODER_DEVICE", globals().get("SENTENCE_EMBEDDING_DEVICE", "cpu"))
        _CROSS_ENCODER_CACHE[model_name] = CrossEncoder(model_name, device=device)
    return _CROSS_ENCODER_CACHE[model_name]


def rerank_chunks_with_cross_encoder(question_text: str, ranked: list[dict], top_k: int = 8):
    """Rerank top chunks with a cross-encoder; return None if unavailable."""
    if not globals().get("USE_CROSS_ENCODER_RERANKER", False) or len(ranked) <= 1:
        return None

    candidate_count = min(len(ranked), max(top_k, globals().get("CROSS_ENCODER_RERANK_TOP_N", 12)))
    cross_weight = float(globals().get("CROSS_ENCODER_RERANK_WEIGHT", 0.55))
    candidates = ranked[:candidate_count]

    try:
        model = load_cross_encoder_model()
        pairs = [(question_text, f"{item.get('title', '')} {item.get('text', '')}") for item in candidates]
        cross_scores = model.predict(pairs)
    except Exception as exc:
        print(f"Cross-encoder reranker skipped, using sentence embedding/BM25 order: {exc}")
        return None

    cross_scores = [float(score) for score in cross_scores]
    min_cross = min(cross_scores, default=0.0)
    max_cross = max(cross_scores, default=1.0)
    cross_span = max(max_cross - min_cross, 1e-9)
    max_retrieval_score = max((float(item.get("retrieval_score", 0.0)) for item in candidates), default=1.0) or 1.0

    reranked = []
    for item, raw_cross_score in zip(candidates, cross_scores):
        enriched = dict(item)
        normalized_retrieval = float(enriched.get("retrieval_score", 0.0)) / max_retrieval_score
        normalized_cross = (raw_cross_score - min_cross) / cross_span
        enriched["cross_encoder_score"] = raw_cross_score
        enriched["pre_rerank_retrieval_score"] = enriched.get("retrieval_score")
        enriched["retrieval_score"] = ((1.0 - cross_weight) * normalized_retrieval) + (cross_weight * normalized_cross)
        enriched["retrieval_method"] = f"{enriched.get('retrieval_method', 'retrieval')}+cross_encoder"
        reranked.append(enriched)

    reranked.sort(key=lambda item: item["retrieval_score"], reverse=True)
    return reranked[:top_k]


def rerank_chunks_with_sentence_embeddings(question_text: str, ranked: list[dict], top_k: int = 8) -> list[dict]:
    """Rerank the strongest lexical/BM25 chunks using semantic sentence similarity."""
    cross_encoder_ranked = rerank_chunks_with_cross_encoder(question_text, ranked, top_k=top_k)
    if cross_encoder_ranked is not None:
        return cross_encoder_ranked

    if not globals().get("USE_SENTENCE_EMBEDDING_RERANKER", True) or len(ranked) <= 1:
        return ranked[:top_k]

    candidate_count = min(len(ranked), max(top_k, globals().get("EMBEDDING_RERANK_TOP_N", 20)))
    embedding_weight = float(globals().get("EMBEDDING_RERANK_WEIGHT", 0.35))
    candidates = ranked[:candidate_count]

    try:

        model = load_sentence_embedding_model()
        chunk_texts = [f"{item.get('title', '')} {item.get('text', '')}" for item in candidates]
        question_embedding = model.encode([question_text], normalize_embeddings=True, convert_to_numpy=True)[0]
        chunk_embeddings = model.encode(chunk_texts, normalize_embeddings=True, convert_to_numpy=True)
        semantic_scores = np.matmul(chunk_embeddings, question_embedding)
    except Exception as exc:
        print(f"Sentence embedding reranker skipped, keeping BM25 order: {exc}")
        return ranked[:top_k]

    max_retrieval_score = max((float(item.get("retrieval_score", 0.0)) for item in candidates), default=1.0) or 1.0
    reranked = []
    for item, semantic_score in zip(candidates, semantic_scores):
        enriched = dict(item)
        normalized_retrieval = float(enriched.get("retrieval_score", 0.0)) / max_retrieval_score
        semantic_score = float(semantic_score)
        enriched["semantic_score"] = semantic_score
        enriched["pre_rerank_retrieval_score"] = enriched.get("retrieval_score")
        enriched["retrieval_score"] = ((1.0 - embedding_weight) * normalized_retrieval) + (embedding_weight * semantic_score)
        enriched["retrieval_method"] = f"{enriched.get('retrieval_method', 'retrieval')}+sentence_embedding"
        reranked.append(enriched)

    reranked.sort(key=lambda item: item["retrieval_score"], reverse=True)
    return reranked[:top_k]


def ensure_pyterrier_started():
    """Import and initialize PyTerrier once for BM25 retrieval."""
    

    started = True
    if hasattr(pt, "started"):
        started = pt.started()
    elif hasattr(pt, "java") and hasattr(pt.java, "started"):
        started = pt.java.started()

    if not started:
        if hasattr(pt, "init"):
            pt.init()
        elif hasattr(pt, "java") and hasattr(pt.java, "init"):
            pt.java.init()

    return pt


def retrieve_rag_chunks_with_tfidf_fallback(question_text: str, chunks: list[dict], top_k: int = 8) -> list[dict]:
    """Fallback retriever used only when PyTerrier is unavailable."""
    chunk_texts = [f"{chunk['title']} {chunk['text']}" for chunk in chunks]

    try:

        vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
        matrix = vectorizer.fit_transform([question_text] + chunk_texts)
        similarities = cosine_similarity(matrix[0:1], matrix[1:]).flatten()
        method = "tfidf_cosine_fallback"
    except Exception:
        similarities = [lexical_similarity(question_text, text) for text in chunk_texts]
        method = "lexical_overlap_fallback"

    ranked = []
    for chunk, similarity in zip(chunks, similarities):
        score = float(similarity) + 0.08 * chunk.get("document_score", 0.0)
        enriched = dict(chunk)
        enriched["retrieval_score"] = score
        enriched["retrieval_method"] = method
        ranked.append(enriched)

    ranked.sort(key=lambda item: item["retrieval_score"], reverse=True)
    return rerank_chunks_with_sentence_embeddings(question_text, ranked, top_k=top_k)


def retrieve_rag_chunks(question, documents: list[dict], top_k: int = 8) -> list[dict]:
    """Retrieve the strongest chunks with PyTerrier BM25 over the Wikipedia chunks."""
    question_text = question_to_text(question)
    bm25_query = " ".join(extract_keywords(question_text, limit=30)) or question_text
    chunks = build_rag_chunks(documents)
    if not chunks:
        return []
    if not globals().get("USE_PYTERRIER_BM25", True):
        return retrieve_rag_chunks_with_tfidf_fallback(question_text, chunks, top_k=top_k)

    try:

        pt = ensure_pyterrier_started()
        index_dir = tempfile.mkdtemp(prefix="pt_rag_chunks_")
        try:
            indexer = pt.IterDictIndexer(index_dir, meta={"docno": 32}, overwrite=True)
            index_ref = indexer.index(
                {
                    "docno": str(index),
                    "text": f"{chunk.get('title', '')} {chunk.get('text', '')}",
                }
                for index, chunk in enumerate(chunks)
            )
            if hasattr(pt, "terrier") and hasattr(pt.terrier, "Retriever"):
                retriever = pt.terrier.Retriever(index_ref, wmodel="BM25", metadata=["docno"])
            else:
                retriever = pt.BatchRetrieve(index_ref, wmodel="BM25", metadata=["docno"])
            results = retriever.search(bm25_query)
        finally:
            shutil.rmtree(index_dir, ignore_errors=True)

        score_by_docno = {
            str(row.docno): float(row.score)
            for row in results.itertuples(index=False)
        }
        max_bm25 = max(score_by_docno.values(), default=0.0)
        if max_bm25 <= 0:
            return retrieve_rag_chunks_with_tfidf_fallback(question_text, chunks, top_k=top_k)

        ranked = []
        for index, chunk in enumerate(chunks):
            raw_bm25 = score_by_docno.get(str(index), 0.0)
            normalized_bm25 = raw_bm25 / max_bm25 if max_bm25 else 0.0
            score = normalized_bm25 + 0.08 * chunk.get("document_score", 0.0)
            enriched = dict(chunk)
            enriched["retrieval_score"] = score
            enriched["bm25_score"] = raw_bm25
            enriched["retrieval_method"] = "pyterrier_bm25"
            ranked.append(enriched)

        ranked.sort(key=lambda item: item["retrieval_score"], reverse=True)
        return rerank_chunks_with_sentence_embeddings(question_text, ranked, top_k=top_k)
    except Exception as exc:
        print(f"PyTerrier BM25 retrieval skipped, using fallback retrieval instead: {exc}")
        return retrieve_rag_chunks_with_tfidf_fallback(question_text, chunks, top_k=top_k)


def build_rag_context(hits: list[dict], max_chars: int = 4200) -> str:
    """Format retrieved chunks as compact evidence for generation."""
    blocks = []
    used = 0
    for index, hit in enumerate(hits, start=1):
        block = f"[Evidence {index} | {hit['title']}] {hit['text']}"
        if used + len(block) > max_chars:
            block = block[:max(0, max_chars - used)]
        if block.strip():
            blocks.append(block)
            used += len(block)
        if used >= max_chars:
            break
    return "\n\n".join(blocks)


def rank_answer_sentences(question, hits: list[dict], max_sentences: int = 5) -> list[str]:
    """Extract the most relevant evidence sentences for a extractive answer."""
    question_text = question_to_text(question)
    question_terms = set(tokenize(question_text))
    purpose_terms = {"goal", "purpose", "reason", "important", "considered", "used", "use", "tool", "primarily", "primary", "fundamental", "institution"}
    wants_purpose = bool(question_terms & purpose_terms)
    candidates = []
    seen = set()

    for hit_index, hit in enumerate(hits):
        for sentence_index, sentence in enumerate(split_sentences(hit.get("text", ""))):
            key = sentence.lower()
            if key in seen:
                continue
            seen.add(key)
            sentence_terms = set(tokenize(sentence))
            sentence_for_score = f"{hit.get('title', '')} {sentence}"
            score = lexical_similarity(question_text, sentence_for_score) + 0.15 * hit.get("retrieval_score", 0.0)
            if wants_purpose:
                score += 0.25 * len(sentence_terms & purpose_terms)
            if hit.get("title", "").lower() in sentence.lower():
                score += 0.05
            candidates.append((score, hit_index, sentence_index, sentence))

    candidates.sort(key=lambda item: item[0], reverse=True)
    return [sentence for _, _, _, sentence in candidates[:max_sentences]]


def extractive_rag_answer(question, hits: list[dict], max_sentences: int = 5) -> str:
    """Create a concise explanation paragraph from retrieved evidence sentences."""
    sentences = rank_answer_sentences(question, hits, max_sentences=max_sentences)
    if not sentences:
        return "I could not find enough evidence in the retrieved Wikipedia documents to answer confidently."
    return " ".join(sentences)


ANSWER_MODEL_ID = globals().get("ANSWER_MODEL_ID", globals().get("MODEL_ID", "Qwen/Qwen2.5-7B-Instruct"))
_ANSWER_MODEL_CACHE = globals().setdefault("_ANSWER_MODEL_CACHE", {})
if "tokenizer" in globals() and "model" in globals():
    _ANSWER_MODEL_CACHE.setdefault(ANSWER_MODEL_ID, (tokenizer, model))

def generate_rag_answer(question, hits: list[dict], model_name: str = ANSWER_MODEL_ID, max_new_tokens: int = 180) -> str:
    """Generate an explanatory RAG answer with the local answer model."""
    

    tokenizer, model = load_answer_model(model_name)
    question_text = question_to_text(question)
    context = build_rag_context(hits)

    messages = [
        {
            "role": "system",
            "content": "Answer the question using only the retrieved evidence. Be concise and do not invent facts.",
        },
        {
            "role": "user",
            "content": f"Evidence:\n{context}\n\nQuestion: {question_text}\n\nAnswer:",
        },
    ]

    if hasattr(tokenizer, "apply_chat_template"):
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        prompt = messages[0]["content"] + "\n\n" + messages[1]["content"] + "\nAnswer:"

    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


def answer_question_with_rag(
    question,
    documents: list[dict],
    top_k_chunks: int = 12,
    use_local_generator: bool = True,
    generator_model: str = ANSWER_MODEL_ID,
) -> dict:
    """
    Ask the question from RAG using the top-N documents, without using answer options.

    Returns an explanatory answer plus the retrieved evidence chunks.
    """
    hits = retrieve_rag_chunks(question, documents, top_k=top_k_chunks)
    method = "extractive_rag"

    if use_local_generator:
        try:
            answer = generate_rag_answer(question, hits, model_name=generator_model)
            method = f"answer_model_rag:{generator_model}"
        except Exception as exc:
            print(f"Local generator skipped, using extractive RAG instead: {exc}")
            answer = extractive_rag_answer(question, hits)
    else:
        answer = extractive_rag_answer(question, hits)

    return {
        "question": question_to_text(question),
        "answer": answer,
        "method": method,
        "evidence_chunks": hits,
    }

### 4c. Answer Model Loader

In [25]:
import torch


def load_answer_model(model_name: str = ANSWER_MODEL_ID):
    """Return the shared answer model loaded in the setup cell.

    All pipelines use the same Qwen model instance. This helper intentionally
    does not call from_pretrained, so later pipeline cells cannot allocate a
    second copy of the 7B model by accident.
    """
    requested_model = model_name or ANSWER_MODEL_ID
    shared_model_id = globals().get("MODEL_ID", ANSWER_MODEL_ID)

    if requested_model in _ANSWER_MODEL_CACHE:
        return _ANSWER_MODEL_CACHE[requested_model]

    if requested_model == shared_model_id and "tokenizer" in globals() and "model" in globals():
        _ANSWER_MODEL_CACHE[requested_model] = (tokenizer, model)
        return tokenizer, model

    raise RuntimeError(
        f"Model {requested_model} is not loaded. Run the shared model loader cell first, "
        f"or change MODEL_ID there to {requested_model} and rerun from setup."
    )

### 4d. Preload & Warm Up

In [26]:
import time
import torch

# Run this BEFORE starting a timed game.
# Hugging Face auth is no longer required for this notebook.



start_time = time.time()
ANSWER_MODEL_FOR_GAME = globals().get("ANSWER_MODEL_FOR_GAME", globals().get("MODEL_ID", "Qwen/Qwen2.5-7B-Instruct"))
print(f"Preloading {ANSWER_MODEL_FOR_GAME} before the timed game...")
try:
    answer_tokenizer, answer_model = load_answer_model(ANSWER_MODEL_FOR_GAME)
except Exception as exc:
    raise RuntimeError(
        f"Could not load answer model {ANSWER_MODEL_FOR_GAME}. Restart the Colab runtime, run only the setup cells, "
        "or choose a smaller model if GPU memory is tight."
    ) from exc
ACTIVE_ANSWER_MODEL_ID = next((name for name, cached in _ANSWER_MODEL_CACHE.items() if cached == (answer_tokenizer, answer_model)), ANSWER_MODEL_FOR_GAME)

print(f"Active model: {ACTIVE_ANSWER_MODEL_ID}")
if globals().get("USE_SENTENCE_EMBEDDING_RERANKER", False):
    print(f"Preloading sentence embedding model {SENTENCE_EMBEDDING_MODEL_ID}...")
    sentence_embedding_model = load_sentence_embedding_model()
if globals().get("USE_CROSS_ENCODER_RERANKER", False):
    print(f"Preloading cross-encoder reranker {CROSS_ENCODER_MODEL_ID}...")
    cross_encoder_model = load_cross_encoder_model()

# Small warm-up generation so first real RAG answer does not pay setup cost.
warmup_messages = [
    {"role": "system", "content": "Answer shortly."},
    {"role": "user", "content": "Say I wanna be a PoliMillionaire."},
]
warmup_prompt = answer_tokenizer.apply_chat_template(warmup_messages, tokenize=False, add_generation_prompt=True)
warmup_device = next(answer_model.parameters()).device
warmup_inputs = answer_tokenizer(warmup_prompt, return_tensors="pt").to(warmup_device)
with torch.inference_mode():
    _ = answer_model.generate(
        **warmup_inputs,
        max_new_tokens=10,
        do_sample=False,
        pad_token_id=answer_tokenizer.eos_token_id,
    )

print(f"Answer and retrieval models are loaded and warmed up in {time.time() - start_time:.1f}s.")
print("Now start the game / run the RAG answer cell.")

Preloading Qwen/Qwen2.5-7B-Instruct before the timed game...
Active model: Qwen/Qwen2.5-7B-Instruct


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Answer and retrieval models are loaded and warmed up in 1.5s.
Now start the game / run the RAG answer cell.


### 4e. MCQ Prompt Builder

In [27]:
SYSTEM_PROMPT = """
You are an expert multiple choice quiz solver.

Carefully analyze the question.

Return ONLY the single best answer option.

Do not explain your reasoning.
Do not output extra text.
"""

def build_mcq_prompt(question):

    choices = []

    for i, opt in enumerate(question.options):
        letter = chr(ord("A") + i)
        choices.append(f"{letter}. {option_text(opt)}")

    joined = "\n".join(choices)

    return f"""
Question:
{question.text}

Options:
{joined}

Reply with ONLY one letter: A, B, C, or D.
"""

### 4f. Direct Logit Scoring & Option Matching

In [28]:
import hashlib
import random
import torch
import torch.nn.functional as F
import numpy as np

LETTERS = "ABCD"


def option_text(option) -> str:
    return option.text if hasattr(option, "text") else option["text"]


def option_id(option) -> int:
    return option.id if hasattr(option, "id") else option["id"]


def parse_option_choice(text: str, option_count: int = 4):
    """Parse A-D or 0-3 from the model output."""
    cleaned = str(text).strip().upper()

    letter_match = re.search(r"\b([A-D])\b", cleaned)
    if letter_match:
        index = LETTERS.index(letter_match.group(1))
        return index if index < option_count else None

    digit_match = re.search(r"\b([0-3])\b", cleaned)
    if digit_match:
        index = int(digit_match.group(1))
        return index if index < option_count else None

    return None


def parse_option_choice_with_text(text: str, options):
    """Parse the chosen option, preferring an exact option-text mention over a possibly wrong letter."""
    normalized_output = normalize_match_text(text)
    text_matches = []
    for index, option in enumerate(options):
        normalized_option = normalize_match_text(option_text(option))
        if normalized_option and normalized_option in normalized_output:
            text_matches.append(index)
    if len(text_matches) == 1:
        return text_matches[0]
    return parse_option_choice(text, option_count=len(options))



def choose_option_direct_shuffled_logits(question, options, model_name: str = ANSWER_MODEL_ID) -> dict:
    """Average next-letter logit scores across shuffled option orders."""

    vote_count = max(3, int(globals().get("DIRECT_MODEL_VOTES", 3)))
    options = list(options)

    question_text = question_to_text(question)
    tokenizer, model = load_answer_model(model_name)
    device = next(model.parameters()).device

    score_lists = {index: [] for index in range(len(options))}
    vote_details = []

    for vote_number in range(vote_count):
        order = list(range(len(options)))

        if vote_number > 0:
            seed = int(
                hashlib.sha256(
                    f"{question_text}|logits|{vote_number}".encode("utf-8")
                ).hexdigest()[:12],
                16,
            )
            rng = random.Random(seed)
            rng.shuffle(order)

            if order == list(range(len(options))) and len(order) > 1:
                order = order[1:] + order[:1]

        option_lines = "\n".join(
            f"{LETTERS[display_index]}. {option_text(options[original_index])}"
            for display_index, original_index in enumerate(order)
        )

        user_prompt = f"""
Question:
{question_text}

Options:
{option_lines}

Reply with ONLY one letter: A, B, C, or D.
"""

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]

        if hasattr(tokenizer, "apply_chat_template"):
            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        else:
            prompt = SYSTEM_PROMPT + "\n\n" + user_prompt

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1536,
        ).to(device)

        with torch.inference_mode():
            outputs = model(**inputs)
            log_probs = F.log_softmax(outputs.logits[0, -1], dim=-1)

        display_scores = []

        for display_index, original_index in enumerate(order):
            letter = LETTERS[display_index]
            variant_scores = []

            for variant in (letter, f" {letter}", f"{letter}.", f" {letter}."):
                token_ids = tokenizer.encode(variant, add_special_tokens=False)
                if token_ids:
                    variant_scores.append(float(log_probs[token_ids[0]].detach().cpu()))

            best_score = max(variant_scores) if variant_scores else float("-inf")

            score_lists[original_index].append(best_score)
            display_scores.append({
                "display_letter": letter,
                "answer_index": original_index,
                "logprob": best_score,
            })

        display_scores.sort(key=lambda item: item["logprob"], reverse=True)

        vote_details.append({
            "vote_number": vote_number + 1,
            "display_order": order,
            "ranked": display_scores,
        })

    averaged_scores = []

    for index, scores in score_lists.items():
        averaged_scores.append({
            "answer_index": index,
            "letter": LETTERS[index],
            "avg_logprob": sum(scores) / max(1, len(scores)),
            "logprobs": scores,
        })

    ranked = sorted(
        averaged_scores,
        key=lambda item: item["avg_logprob"],
        reverse=True,
    )

    selected_index = ranked[0]["answer_index"]
    margin = ranked[0]["avg_logprob"] - ranked[1]["avg_logprob"] if len(ranked) > 1 else 0.0
    selected_option = options[selected_index]

    return {
        "answer_id": option_id(selected_option),
        "answer_text": option_text(selected_option),
        "answer_index": selected_index,
        "letter": LETTERS[selected_index],
        "model_output": "shuffled_logit_scores:" + ", ".join(
            f"{item['letter']}={item['avg_logprob']:.3f}" for item in ranked
        ),
        "selection_source": f"direct_model_shuffled_logit_scoring_{vote_count}",
        "direct_logit_scores": ranked,
        "direct_logit_margin": margin,
        "direct_votes": vote_details,
        "option_scores": [],
    }


def choose_option_direct_voted(question, options, model_name: str = ANSWER_MODEL_ID) -> dict:
    """Fast direct answer with strict MCQ voting."""
    vote_count = int(globals().get("DIRECT_MODEL_VOTES", 1))
    vote_count = max(1, vote_count)

    tokenizer, model = load_answer_model(model_name)
    user_prompt = build_mcq_prompt(question)


    device = next(model.parameters()).device
    votes = []

    for _ in range(vote_count):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]

        if hasattr(tokenizer, "apply_chat_template"):
            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        else:
            prompt = SYSTEM_PROMPT + "\n\n" + user_prompt

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1536,
        ).to(device)

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=2,
                do_sample=False,
                temperature=0.0,
                top_p=1.0,
                pad_token_id=tokenizer.eos_token_id,
            )

        generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
        model_output = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

        selected_index = parse_option_choice_with_text(model_output, options)

        if selected_index is not None:
            votes.append({
                "answer_index": selected_index,
                "letter": LETTERS[selected_index],
                "model_output": model_output,
            })

    if not votes:
        raise ValueError("Could not parse any direct model vote")

    counts = {}
    first_seen = {}

    for order, vote in enumerate(votes):
        index = vote["answer_index"]
        counts[index] = counts.get(index, 0) + 1
        first_seen.setdefault(index, order)

    selected_index = sorted(
        counts,
        key=lambda index: (-counts[index], first_seen[index]),
    )[0]

    selected_option = options[selected_index]

    return {
        "answer_id": option_id(selected_option),
        "answer_text": option_text(selected_option),
        "answer_index": selected_index,
        "letter": LETTERS[selected_index],
        "model_output": " | ".join(vote["model_output"] for vote in votes),
        "selection_source": f"direct_model_voted_{len(votes)}",
        "direct_votes": votes,
        "option_scores": [],
    }





def normalize_match_text(text: str) -> str:
    """Normalize text for cheap exact/near-exact option matching."""
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", str(text).lower())).strip()


def score_options_against_evidence(question, options, hits: list[dict]) -> list[dict]:
    """Score each option directly against retrieved evidence using fast lexical/exact matching."""
    question_text = question_to_text(question)
    evidence_texts = [f"{hit.get('title', '')} {hit.get('text', '')}" for hit in hits if hit.get("text")]
    evidence_blob = " ".join(evidence_texts)
    normalized_evidence = normalize_match_text(evidence_blob)
    question_terms = set(extract_keywords(question_text, limit=24))
    evidence_sentences = split_sentences(evidence_blob) if evidence_blob else []
    try:
        core_terms = core_question_terms(question)
    except NameError:
        core_terms = []
    evidence_terms = set(tokenize(evidence_blob)) if evidence_blob else set()
    evidence_core_overlap = len([term for term in core_terms if expand_term(term) & evidence_terms]) / max(1, len(core_terms)) if core_terms else 1.0
    topic_relevance_multiplier = 1.0 if evidence_core_overlap > 0 else 0.35
    scored = []

    lexical_scores = []
    for option in options:
        option_query = f"{question_text} {option_text(option)}"
        if evidence_texts:
            lexical_scores.append(max(lexical_similarity(option_query, evidence_text) for evidence_text in evidence_texts))
        else:
            lexical_scores.append(0.0)

    semantic_scores = [0.0 for _ in options]
    if evidence_texts and globals().get("USE_OPTION_EMBEDDING_SCORER", True):
        try:

            model = load_sentence_embedding_model()
            option_queries = [f"{question_text} {option_text(option)}" for option in options]
            option_embeddings = model.encode(option_queries, normalize_embeddings=True, convert_to_numpy=True)
            evidence_embeddings = model.encode(evidence_texts, normalize_embeddings=True, convert_to_numpy=True)
            similarities = np.matmul(option_embeddings, evidence_embeddings.T)
            semantic_scores = [float(row.max()) for row in similarities]
        except Exception as exc:
            print(f"Option embedding scorer skipped, using lexical option scores only: {exc}")

    for index, option in enumerate(options):
        lexical_score = float(lexical_scores[index])
        semantic_score = float(semantic_scores[index])
        normalized_option = normalize_match_text(option_text(option))
        option_terms = [term for term in normalized_option.split() if len(term) > 2]
        exact_match = bool(normalized_option and normalized_option in normalized_evidence)
        support_sentence_score = 0.0
        support_sentence = ""
        for sentence in evidence_sentences:
            normalized_sentence = normalize_match_text(sentence)
            if not normalized_option or normalized_option not in normalized_sentence:
                continue
            sentence_terms = set(tokenize(sentence))
            overlap = len(question_terms & sentence_terms) / max(1, len(question_terms))
            score = 0.75 + overlap
            if "only once" in normalized_sentence or "rarely" in normalized_sentence:
                score -= 0.75
            if score > support_sentence_score:
                support_sentence_score = score
                support_sentence = sentence[:260]
        term_coverage = len([term for term in option_terms if term in normalized_evidence]) / max(1, len(option_terms))
        exact_boost = float(globals().get("EXACT_OPTION_MATCH_BOOST", 1.8)) if exact_match else 0.0
        coverage_boost = float(globals().get("OPTION_TERM_COVERAGE_WEIGHT", 0.35)) * term_coverage
        combined_score = ((0.45 * lexical_score) + (0.55 * semantic_score) + exact_boost + coverage_boost + support_sentence_score) * topic_relevance_multiplier
        scored.append(
            {
                "answer_id": option_id(option),
                "answer_text": option_text(option),
                "answer_index": index,
                "letter": LETTERS[index],
                "lexical_score": lexical_score,
                "semantic_score": semantic_score,
                "exact_match": exact_match,
                "term_coverage": term_coverage,
                "support_sentence_score": support_sentence_score,
                "support_sentence": support_sentence,
                "evidence_core_overlap": evidence_core_overlap,
                "combined_score": combined_score,
            }
        )

    scored.sort(key=lambda item: item["combined_score"], reverse=True)
    return scored




### 4g. Full RAG + Agentic Tool Router Pipeline

In [29]:
def is_numeric_question(question) -> bool:
    q = question.text.lower()
    option_texts = " ".join(option_text(opt) for opt in question.options)

    numeric_words = [
        "population", "year", "date", "century", "how many",
        "number", "estimated", "amount", "percentage", "million",
        "billion", "km", "meters", "age"
    ]

    has_digit_option = bool(re.search(r"\d", option_texts))
    has_numeric_word = any(word in q for word in numeric_words)

    return has_digit_option or has_numeric_word


def answer_one_question_with_pipeline(question, game=None) -> dict:
    start = time.monotonic()
    seconds_left_start = seconds_available(game) if game is not None else None
    direct_option_match = None
    direct_model_seconds = 0.0

    if seconds_left_start is not None and seconds_left_start < MIN_SECONDS_FOR_ANY_MODEL:
        selected = fallback_option(question)
        return {
            "question": question.text,
            "options": [{"id": option_id(opt), "text": option_text(opt)} for opt in question.options],
            "documents": [],
            "rag_answer": "Skipped pipeline because too little time remained.",
            "rag_method": "hard_time_guard",
            "evidence_chunks": [],
            "option_match": {
                "answer_id": option_id(selected),
                "answer_text": option_text(selected),
                "answer_index": 0,
                "letter": "A",
                "model_output": "hard_time_guard",
                "selection_source": "hard_time_guard",
                "option_scores": [],
            },
            "elapsed_seconds": 0.0,
            "timings": {},
            "seconds_left_start": seconds_left_start,
            "seconds_left_end": seconds_left_start,
        }

    if USE_DIRECT_MODEL_FIRST and (
        seconds_left_start is None
        or seconds_left_start >= QUESTION_TIME_BUFFER + MIN_SECONDS_FOR_DIRECT_MODEL
    ):
        direct_start = time.monotonic()
        try:
            if USE_DIRECT_LOGIT_SCORING:
                direct_option_match = choose_option_direct_shuffled_logits(
                    question,
                    question.options,
                    model_name=ANSWER_MODEL_FOR_GAME,
                )
                if direct_option_match.get("direct_logit_margin", 0.0) < DIRECT_LOGIT_CONFIDENCE_MARGIN:
                    print(
                        f"Low direct logit margin "
                        f"({direct_option_match.get('direct_logit_margin', 0.0):.3f}); trying prompt voting."
                    )
                    voted_match = choose_option_direct_voted(
                        question,
                        question.options,
                        model_name=ANSWER_MODEL_FOR_GAME,
                    )
                    voted_match["logit_match"] = direct_option_match
                    direct_option_match = voted_match
            else:
                direct_option_match = choose_option_direct_voted(
                    question,
                    question.options,
                    model_name=ANSWER_MODEL_FOR_GAME,
                )

            direct_end = time.monotonic()
            direct_model_seconds = direct_end - direct_start

        except Exception as exc:
            print(f"Direct model answer failed; falling back to Wikipedia: {exc}")


    if direct_option_match is not None:
        current_seconds_left = seconds_available(game) if game is not None else seconds_left_start
        direct_margin = direct_logit_margin(direct_option_match)
        use_tool_router = globals().get("USE_AGENTIC_TOOL_ROUTER", True)
        skip_margin = float(globals().get("DIRECT_TOOL_SKIP_MARGIN", DIRECT_LOGIT_CONFIDENCE_MARGIN))
        verify_direct = globals().get("VERIFY_DIRECT_WITH_WIKIPEDIA", True)
        has_tool_time = current_seconds_left is None or current_seconds_left >= QUESTION_TIME_BUFFER + MIN_SECONDS_TO_VERIFY_DIRECT
        high_confidence_direct = use_tool_router and direct_margin >= skip_margin
        should_skip_tools = high_confidence_direct or not verify_direct or not has_tool_time

        if should_skip_tools:
            if high_confidence_direct:
                router_reason = f"high_direct_margin:{direct_margin:.3f}>={skip_margin:.3f}"
            elif not verify_direct:
                router_reason = "verification_disabled"
            else:
                router_reason = "not_enough_time_for_tool_call"
            direct_option_match["selection_source"] = "agentic_router_direct_answer"
            direct_option_match["tool_router_reason"] = router_reason
            direct_option_match["direct_logit_margin"] = direct_margin
            after_match = time.monotonic()
            return {
                "question": question.text,
                "options": [{"id": option_id(opt), "text": option_text(opt)} for opt in question.options],
                "documents": [],
                "rag_answer": f"Tool router skipped live Wikipedia ({router_reason}); used direct {ANSWER_MODEL_FOR_GAME} answer.",
                "rag_method": "agentic_router_direct_answer",
                "evidence_chunks": [],
                "option_match": direct_option_match,
                "elapsed_seconds": after_match - start,
                "timings": {
                    "direct_model_seconds": direct_model_seconds,
                    "wikipedia_seconds": 0.0,
                    "rag_generation_seconds": 0.0,
                    "option_matching_seconds": after_match - start - direct_model_seconds,
                    "pre_submit_pipeline_seconds": after_match - start,
                },
                "seconds_left_start": seconds_left_start,
                "seconds_left_end": seconds_available(game) if game is not None else None,
            }

        print(f"Tool router: direct answer was low confidence (margin={direct_margin:.3f}); calling Wikipedia tool path.")

    current_seconds_left = seconds_available(game) if game is not None else seconds_left_start
    if current_seconds_left is not None:
        final_model_reserve = MIN_SECONDS_FOR_FINAL_MODEL
        retrieval_budget = max(0.0, current_seconds_left - QUESTION_TIME_BUFFER - final_model_reserve)
        retrieval_budget = min(WIKIPEDIA_TIME_BUDGET, retrieval_budget)
    else:
        retrieval_budget = WIKIPEDIA_TIME_BUDGET

    retrieval_deadline = time.monotonic() + max(0.0, retrieval_budget)

    if USE_LIVE_WIKIPEDIA:
        docs = get_wikipedia_documents_for_question(
            question,
            top_n=TOP_N_DOCS,
            per_query_limit=PER_QUERY_LIMIT,
            timeout=WIKIPEDIA_TIMEOUT,
            max_search_queries=MAX_SEARCH_QUERIES,
            options=question.options,
            deadline_monotonic=retrieval_deadline,
        )
    else:
        docs = []

    after_wikipedia = time.monotonic()
    seconds_left_after_wiki = seconds_available(game) if game is not None else None

    if seconds_left_after_wiki is not None and seconds_left_after_wiki <= QUESTION_TIME_BUFFER:
        if direct_option_match is not None:
            option_match = direct_option_match
            option_match["selection_source"] = "direct_model_after_wikipedia_time_guard"
        else:
            option_match = score_fallback_option_match(
                question,
                [],
                "submit_time_guard_after_wikipedia",
            )
        after_match = time.monotonic()
        return {
            "question": question.text,
            "options": [{"id": option_id(opt), "text": option_text(opt)} for opt in question.options],
            "documents": [
                {
                    "title": doc.get("title"),
                    "url": doc.get("url"),
                    "candidate_score": doc.get("candidate_score"),
                    "matched_query": doc.get("matched_query"),
                }
                for doc in docs
            ],
            "rag_answer": "Skipped chunk retrieval because submit buffer was reached.",
            "rag_method": "submit_time_guard_after_wikipedia",
            "evidence_chunks": [],
            "option_match": option_match,
            "elapsed_seconds": after_match - start,
            "timings": {
                "wikipedia_seconds": after_wikipedia - start,
                "pre_submit_pipeline_seconds": after_match - start,
            },
            "seconds_left_start": seconds_left_start,
            "seconds_left_end": seconds_available(game) if game is not None else None,
        }

    if not docs:
        if direct_option_match is not None:
            option_match = direct_option_match
            option_match["selection_source"] = "direct_model_no_wikipedia_docs"
        else:
            option_match = choose_option_direct_voted(
                question,
                question.options,
                model_name=ANSWER_MODEL_FOR_GAME,
            )
            option_match["selection_source"] = "direct_model_no_wikipedia_docs_retry"

        after_match = time.monotonic()
        if direct_option_match is not None:
            evidence_scores = option_match.get("option_scores", [])
            evidence_is_strong = False

            if evidence_scores and len(evidence_scores) >= 2:
                top = float(evidence_scores[0].get("combined_score", 0.0))
                second = float(evidence_scores[1].get("combined_score", 0.0))
                evidence_is_strong = (
                top >= MIN_EVIDENCE_SCORE_TO_TRUST
                and (top - second) >= EVIDENCE_CONFIDENCE_MARGIN
                )

            if not evidence_is_strong:
                option_match = direct_option_match
                option_match["selection_source"] = "direct_model_preferred_over_weak_rag"
        return {
            "question": question.text,
            "options": [{"id": option_id(opt), "text": option_text(opt)} for opt in question.options],
            "documents": [],
            "rag_answer": f"No Wikipedia documents; used direct {ANSWER_MODEL_FOR_GAME} answer.",
            "rag_method": "no_docs_direct_model",
            "evidence_chunks": [],
            "option_match": option_match,
            "elapsed_seconds": after_match - start,
            "timings": {
                "direct_model_seconds": direct_model_seconds,
                "wikipedia_seconds": after_wikipedia - start - direct_model_seconds,
                "pre_submit_pipeline_seconds": after_match - start,
            },
            "seconds_left_start": seconds_left_start,
            "seconds_left_end": seconds_available(game) if game is not None else None,
     }

    seconds_left_before_rag = seconds_available(game) if game is not None else None
    allow_rag_generator = GENERATE_RAG_DRAFT and (
        seconds_left_before_rag is None
        or seconds_left_before_rag >= QUESTION_TIME_BUFFER + MIN_SECONDS_FOR_FINAL_MODEL
    )

    rag_result = answer_question_with_rag(
        question,
        docs,
        top_k_chunks=TOP_K_CHUNKS,
        use_local_generator=allow_rag_generator,
        generator_model=ANSWER_MODEL_FOR_GAME,
    )

    after_rag = time.monotonic()
    seconds_left_after_rag = seconds_available(game) if game is not None else None

    evidence_hits_for_scoring = [{"title": "RAG answer", "text": rag_result.get("answer", "")}] + rag_result["evidence_chunks"]

    evidence_match = score_fallback_option_match(
        question,
        evidence_hits_for_scoring,
        "wikipedia_evidence_score",
    )

    evidence_scores = evidence_match.get("option_scores", [])
    evidence_is_strong = False

    if evidence_scores and len(evidence_scores) >= 2:
        top = float(evidence_scores[0].get("combined_score", 0.0))
        second = float(evidence_scores[1].get("combined_score", 0.0))
        evidence_is_strong = (
            top >= MIN_EVIDENCE_SCORE_TO_TRUST
            and (top - second) >= EVIDENCE_CONFIDENCE_MARGIN
        )


    if direct_option_match is not None and is_numeric_question(question):
        option_match = direct_option_match
        option_match["selection_source"] = "direct_model_numeric_question"

    elif direct_option_match is not None:
        option_match = reconcile_direct_and_evidence(direct_option_match, evidence_match)

    elif evidence_is_strong:
        option_match = evidence_match
        option_match["selection_source"] = "strong_wikipedia_evidence"

    else:
        option_match = evidence_match
        option_match["selection_source"] = "fallback_wikipedia_evidence"

    after_match = time.monotonic()
    elapsed = after_match - start

    return {
        "question": question.text,
        "options": [{"id": option_id(opt), "text": option_text(opt)} for opt in question.options],
        "documents": [
            {
                "title": doc.get("title"),
                "url": doc.get("url"),
                "candidate_score": doc.get("candidate_score"),
                "matched_query": doc.get("matched_query"),
            }
            for doc in docs
        ],
        "rag_answer": rag_result["answer"],
        "rag_method": rag_result["method"],
        "evidence_chunks": [
            {
                "title": hit.get("title"),
                "retrieval_score": hit.get("retrieval_score"),
                "bm25_score": hit.get("bm25_score"),
                "semantic_score": hit.get("semantic_score"),
                "cross_encoder_score": hit.get("cross_encoder_score"),
                "pre_rerank_retrieval_score": hit.get("pre_rerank_retrieval_score"),
                "retrieval_method": hit.get("retrieval_method"),
                "text": hit.get("text", "")[:900],
            }
            for hit in rag_result["evidence_chunks"][:5]
        ],
        "option_match": option_match,
        "elapsed_seconds": elapsed,
        "timings": {
            "direct_model_seconds": direct_model_seconds,
            "wikipedia_seconds": after_wikipedia - start - direct_model_seconds,
            "rag_generation_seconds": after_rag - after_wikipedia,
            "option_matching_seconds": after_match - after_rag,
            "pre_submit_pipeline_seconds": elapsed,
        },
        "seconds_left_start": seconds_left_start,
        "seconds_left_end": seconds_available(game) if game is not None else None,
    }

### 4h. Run History Game

In [33]:
import json
import os
import time
import types
from datetime import datetime, timezone
from pathlib import Path
import torch

# =========================
# ACTUAL GAME: free-Colab direct model + optional RAG verification
# =========================
# Run setup/function cells first: imports/login/client, cell 8, cell 10, cell 13.
# This cell preloads the answer model before starting the timed game, then starts the game.


 # ---- Hyperparameters ----
RUN_ACTUAL_GAME = True
COMPETITION_ID = comp_id
MAX_QUESTIONS = None          # Use 1 or 2 for a small test; None = play until game over.
SUBMIT_ANSWERS = True         # True = send answers to API. False = dry run, no submission.

MAX_SEARCH_QUERIES = 2        # Free Colab/timed game: keep live Wikipedia small to avoid 429s.
PER_QUERY_LIMIT = 2           # Enough fallback candidates without burning the whole timer.
TOP_N_DOCS = 3                # Smaller evidence set for a 30-second question window.



WIKIPEDIA_TIMEOUT = 2.0       # Seconds per Wikipedia API request.
WIKIPEDIA_DELAY_SECONDS = 1.0 # Respect Wikipedia's rate limit.
WIKIPEDIA_BACKOFF_SECONDS = 0.5
WIKIPEDIA_RETRIES = 0
MIN_WIKIPEDIA_CANDIDATE_SCORE = 0.25 # Fetch more plausible pages; evidence scorer filters them later.
TOP_K_CHUNKS = 12          # Keep the final evidence set small for the 30-second timer.
USE_LIVE_WIKIPEDIA = True    # Call Wikipedia during the timed game.
USE_PYTERRIER_BM25 = True      # Timed game: TF-IDF fallback is much faster for small per-question chunks.
USE_SENTENCE_EMBEDDING_RERANKER = True # Free Colab: avoid CPU embedding latency during timed play.
USE_OPTION_EMBEDDING_SCORER = True    # Timed game: use fast lexical option scores unless you have time.
USE_CROSS_ENCODER_RERANKER = True # Timed game: cross-encoder was slower/worse in practice.
USE_DIRECT_MODEL_FIRST = True   # Cheap first opinion; retrieved evidence can verify or correct it.
USE_DIRECT_LOGIT_SCORING = True    # Use stable direct next-token logit scoring for no-evidence fallback.
DIRECT_LOGIT_CONFIDENCE_MARGIN = 1.0 # Low margin triggers vote/Wikipedia fallback.
USE_AGENTIC_TOOL_ROUTER = True # Direct answer first; call Wikipedia only when confidence/time says it is worth it.
DIRECT_TOOL_SKIP_MARGIN = 1.25 # If direct logit margin reaches this, skip tool calls and submit.
VERIFY_DIRECT_WITH_WIKIPEDIA = True # Router may use Wikipedia for low-confidence direct answers.
DIRECT_MODEL_VOTES = 3        # Used only by optional generated-vote helpers, not the default fallback.
GENERATE_RAG_DRAFT = False  # Timed game: avoid slow explanatory generation before choosing an option.
SENTENCE_EMBEDDING_DEVICE = "cpu" # Keep GPU memory for the answer model; use "cuda" only if you have room.
CROSS_ENCODER_DEVICE = "cpu"
EMBEDDING_RERANK_TOP_N = 20    # Rerank this many top BM25/fallback chunks semantically.
EMBEDDING_RERANK_WEIGHT = 0.35 # Higher means semantic similarity influences ranking more.
CROSS_ENCODER_RERANK_TOP_N = 4
CROSS_ENCODER_RERANK_WEIGHT = 0.55
QUESTION_TIME_BUFFER = 6.0    # Submit before this many seconds remain.
MIN_SECONDS_TO_ATTEMPT = 4.0  # If less time remains, submit fallback option 0.
WIKIPEDIA_TIME_BUDGET = 5.0   # Hard cap for Wikipedia search + page fetch.
MIN_SECONDS_FOR_FINAL_MODEL = 8.0 # Require this much extra time, after the submit buffer, before final answer model.
MIN_SECONDS_FOR_ANY_MODEL = 3.0   # Below this, submit fallback immediately.
MIN_SECONDS_FOR_DIRECT_MODEL = 4.0
MIN_SECONDS_TO_VERIFY_DIRECT = 10.0
EVIDENCE_OVERRIDE_MARGIN = 0.20
EVIDENCE_CONFIDENCE_MARGIN = 0.12
MIN_EVIDENCE_SCORE_TO_TRUST = 0.45
MIN_DIRECT_MARGIN_FOR_WEAK_EVIDENCE = 2.50
EXACT_OPTION_MATCH_BOOST = 1.8
OPTION_TERM_COVERAGE_WEIGHT = 0.35

DELAY_SUBMIT_FOR_WIKI_COOLDOWN = False # Never wait on purpose inside a 30-second question.
TARGET_SUBMIT_ELAPSED_SECONDS = 22.0
MIN_SECONDS_LEFT_AT_SUBMIT = 5.0      # Safety margin for network/server latency.
MAX_SUBMIT_WAIT_SECONDS = 0

ANSWER_MODEL_FOR_GAME = globals().get("ANSWER_MODEL_FOR_GAME", globals().get("MODEL_ID", "Qwen/Qwen2.5-7B-Instruct"))
PRELOAD_ANSWER_MODEL = True
SAVE_RUN_LOG = True
RUN_LOG_DIR = "/content/gdrive/MyDrive/NLP_assignment/test3_rag_game_runs"
VERBOSE = True

# ---- End hyperparameters ----






def preload_answer_model_for_game():
    globals()["WIKIPEDIA_REQUEST_DELAY_SECONDS"] = WIKIPEDIA_DELAY_SECONDS
    globals()["WIKIPEDIA_429_BACKOFF_SECONDS"] = WIKIPEDIA_BACKOFF_SECONDS
    globals()["WIKIPEDIA_MAX_RETRIES"] = WIKIPEDIA_RETRIES
    globals()["MAX_WIKIPEDIA_SEARCH_QUERIES"] = MAX_SEARCH_QUERIES
    globals()["USE_PYTERRIER_BM25"] = USE_PYTERRIER_BM25
    print(f"Preloading {ANSWER_MODEL_FOR_GAME} before starting timed game...")
    start = time.time()
    try:
        tokenizer, model = load_answer_model(ANSWER_MODEL_FOR_GAME)
    except Exception as exc:
        raise RuntimeError(
            f"Could not load answer model {ANSWER_MODEL_FOR_GAME}. Restart the Colab runtime, run only the setup cells, "
            "or choose a smaller model if GPU memory is tight. The timed game was not started."
        ) from exc
    active_model_name = next((name for name, cached in _ANSWER_MODEL_CACHE.items() if cached == (tokenizer, model)), ANSWER_MODEL_FOR_GAME)
    globals()["ANSWER_MODEL_FOR_GAME"] = active_model_name
    model_device = next(model.parameters()).device
    model_dtype = next(model.parameters()).dtype
    print(f"Loaded model: {ANSWER_MODEL_FOR_GAME} | device={model_device} | dtype={model_dtype}")


    warmup_messages = [
        {"role": "system", "content": "Answer briefly."},
        {"role": "user", "content": "Say ready."},
    ]
    warmup_prompt = tokenizer.apply_chat_template(warmup_messages, tokenize=False, add_generation_prompt=True)
    device = next(model.parameters()).device
    inputs = tokenizer(warmup_prompt, return_tensors="pt").to(device)
    with torch.inference_mode():
        _ = model.generate(
            **inputs,
            max_new_tokens=2,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    print(f"Direct model ready in {time.time() - start:.1f}s. Starting game only after this point.")
    if globals().get("USE_SENTENCE_EMBEDDING_RERANKER", True):
        print(f"Preloading sentence embedding model {SENTENCE_EMBEDDING_MODEL_ID}...")
        load_sentence_embedding_model()
        print("Sentence embedding reranker ready.")
    if globals().get("USE_CROSS_ENCODER_RERANKER", False):
        print(f"Preloading cross-encoder reranker {CROSS_ENCODER_MODEL_ID}...")
        load_cross_encoder_model()
        print("Cross-encoder reranker ready.")


def seconds_available(game) -> float:
    remaining = game.time_remaining
    if remaining is None:
        return 30.0
    return max(0.0, float(remaining))


def fallback_option(question):
    return question.options[0]


def score_fallback_option_match(question, hits: list[dict], reason: str) -> dict:
    """Choose the highest evidence-score option without another answer-model call."""
    option_scores = score_options_against_evidence(question, question.options, hits)
    best_score = option_scores[0] if option_scores else {"answer_index": 0}
    selected_option = question.options[best_score["answer_index"]]
    return {
        "answer_id": option_id(selected_option),
        "answer_text": option_text(selected_option),
        "answer_index": best_score["answer_index"],
        "letter": LETTERS[best_score["answer_index"]],
        "model_output": reason,
        "selection_source": "timed_option_evidence_score_fallback",
        "selected_option_score": best_score,
        "option_scores": option_scores,
    }


def reconcile_direct_and_evidence(direct_match: dict, evidence_match: dict) -> dict:
    """Keep direct model unless evidence strongly supports another option."""
    if not direct_match:
        return evidence_match
    if not evidence_match or not evidence_match.get("option_scores"):
        direct = dict(direct_match)
        direct["selection_source"] = "direct_model_no_evidence"
        return direct

    scores = evidence_match["option_scores"]
    top = scores[0]
    direct_score = next((item for item in scores if item["answer_index"] == direct_match["answer_index"]), None)
    direct_value = float(direct_score.get("combined_score", 0.0)) if direct_score else 0.0
    top_value = float(top.get("combined_score", 0.0))
    second_value = float(scores[1].get("combined_score", 0.0)) if len(scores) > 1 else 0.0
    evidence_margin = top_value - second_value
    margin = top_value - direct_value
    direct_margin = direct_logit_margin(direct_match)
    direct_is_high_confidence = direct_margin >= DIRECT_LOGIT_CONFIDENCE_MARGIN
    direct_is_safe_with_weak_evidence = direct_margin >= globals().get("MIN_DIRECT_MARGIN_FOR_WEAK_EVIDENCE", 1.20)
    top_exact_with_margin = bool(top.get("exact_match")) and evidence_margin >= EVIDENCE_CONFIDENCE_MARGIN
    top_is_trustworthy = top_exact_with_margin or (
        top_value >= MIN_EVIDENCE_SCORE_TO_TRUST and evidence_margin >= EVIDENCE_OVERRIDE_MARGIN
    )
    should_override = top["answer_index"] != direct_match["answer_index"] and top_is_trustworthy and margin >= EVIDENCE_OVERRIDE_MARGIN

    if should_override:
        chosen = dict(evidence_match)
        chosen["selection_source"] = "wikipedia_evidence_overrode_direct_model"
        chosen["direct_model_match"] = direct_match
        chosen["evidence_override_margin"] = margin
        chosen["evidence_score_margin"] = evidence_margin
        return chosen

    chosen = dict(direct_match)
    if top["answer_index"] == direct_match["answer_index"] and top_is_trustworthy:
        chosen["selection_source"] = "direct_model_confirmed_by_wikipedia_evidence"
    elif direct_is_safe_with_weak_evidence:
        chosen["selection_source"] = "direct_model_high_confidence_weak_evidence"
    elif direct_is_high_confidence:
        chosen["selection_source"] = "direct_model_acceptable_margin_weak_evidence"
    else:
        chosen = dict(evidence_match)
        chosen["selection_source"] = "weak_direct_model_deferred_to_evidence_score"
        chosen["direct_model_match"] = direct_match
        chosen["evidence_override_margin"] = margin
        chosen["evidence_score_margin"] = evidence_margin
        return chosen
    chosen["option_scores"] = scores
    chosen["selected_option_score"] = direct_score
    chosen["evidence_top_option"] = top
    chosen["evidence_override_margin"] = margin
    chosen["evidence_score_margin"] = evidence_margin
    chosen["direct_logit_margin"] = direct_margin
    return chosen


def direct_logit_margin(direct_match: dict) -> float:
    """Return the direct logit margin, including when a generated vote wrapped it."""
    if not direct_match:
        return 0.0
    if "direct_logit_margin" in direct_match:
        return float(direct_match.get("direct_logit_margin", 0.0))
    logit_match = direct_match.get("logit_match") or {}
    return float(logit_match.get("direct_logit_margin", 0.0))


def wait_before_submit_for_cooldown(game) -> float:
    """Wait after computing the answer so Wikipedia gets cooldown time before next question."""
    if not DELAY_SUBMIT_FOR_WIKI_COOLDOWN:
        return 0.0

    current_remaining = seconds_available(game)
    target_remaining = max(MIN_SECONDS_LEFT_AT_SUBMIT, 30.0 - TARGET_SUBMIT_ELAPSED_SECONDS)
    wait_seconds = current_remaining - target_remaining
    wait_seconds = min(MAX_SUBMIT_WAIT_SECONDS, max(0.0, wait_seconds))

    if wait_seconds > 0:
        print(f"Answer ready. Waiting {wait_seconds:.1f}s before submit to give Wikipedia API cooldown time.")
        time.sleep(wait_seconds)

    return wait_seconds



def play_actual_rag_game():
    globals()["WIKIPEDIA_REQUEST_DELAY_SECONDS"] = WIKIPEDIA_DELAY_SECONDS
    globals()["WIKIPEDIA_429_BACKOFF_SECONDS"] = WIKIPEDIA_BACKOFF_SECONDS
    globals()["WIKIPEDIA_MAX_RETRIES"] = WIKIPEDIA_RETRIES
    globals()["MAX_WIKIPEDIA_SEARCH_QUERIES"] = MAX_SEARCH_QUERIES
    globals()["USE_PYTERRIER_BM25"] = USE_PYTERRIER_BM25

    if PRELOAD_ANSWER_MODEL:
        preload_answer_model_for_game()

    # Keep the Wikipedia cooldown state across setup/game questions so live requests do not trip 429s.

    if not RUN_ACTUAL_GAME:
        print("RUN_ACTUAL_GAME is False. Set it to True to start a real timed game.")
        return None, None

    game = client.game.start(competition_id=COMPETITION_ID)
    run_log = {
        "session_id": game.session_id,
        "competition_id": COMPETITION_ID,
        "started_at": datetime.now(timezone.utc).isoformat(),
        "hyperparameters": {
            "top_n_docs": TOP_N_DOCS,
            "per_query_limit": PER_QUERY_LIMIT,
            "max_search_queries": MAX_SEARCH_QUERIES,
            "wikipedia_timeout": WIKIPEDIA_TIMEOUT,
            "wikipedia_delay_seconds": WIKIPEDIA_DELAY_SECONDS,
            "wikipedia_backoff_seconds": WIKIPEDIA_BACKOFF_SECONDS,
            "wikipedia_retries": WIKIPEDIA_RETRIES,
            "min_wikipedia_candidate_score": MIN_WIKIPEDIA_CANDIDATE_SCORE,
            "use_live_wikipedia": USE_LIVE_WIKIPEDIA,
            "use_pyterrier_bm25": USE_PYTERRIER_BM25,
            "top_k_chunks": TOP_K_CHUNKS,
            "use_option_embedding_scorer": USE_OPTION_EMBEDDING_SCORER,
            "use_cross_encoder_reranker": USE_CROSS_ENCODER_RERANKER,
            "cross_encoder_model": CROSS_ENCODER_MODEL_ID,
            "use_direct_model_first": USE_DIRECT_MODEL_FIRST,
            "use_direct_logit_scoring": USE_DIRECT_LOGIT_SCORING,
            "direct_logit_confidence_margin": DIRECT_LOGIT_CONFIDENCE_MARGIN,
            "use_agentic_tool_router": USE_AGENTIC_TOOL_ROUTER,
            "direct_tool_skip_margin": DIRECT_TOOL_SKIP_MARGIN,
            "verify_direct_with_wikipedia": VERIFY_DIRECT_WITH_WIKIPEDIA,
            "direct_model_votes": DIRECT_MODEL_VOTES,
            "min_seconds_for_direct_model": MIN_SECONDS_FOR_DIRECT_MODEL,
            "min_seconds_to_verify_direct": MIN_SECONDS_TO_VERIFY_DIRECT,
            "evidence_override_margin": EVIDENCE_OVERRIDE_MARGIN,
            "evidence_confidence_margin": EVIDENCE_CONFIDENCE_MARGIN,
            "min_evidence_score_to_trust": MIN_EVIDENCE_SCORE_TO_TRUST,
            "min_direct_margin_for_weak_evidence": MIN_DIRECT_MARGIN_FOR_WEAK_EVIDENCE,
            "exact_option_match_boost": EXACT_OPTION_MATCH_BOOST,
            "option_term_coverage_weight": OPTION_TERM_COVERAGE_WEIGHT,
            "generate_rag_draft": GENERATE_RAG_DRAFT,
            "question_time_buffer": QUESTION_TIME_BUFFER,
            "min_seconds_to_attempt": MIN_SECONDS_TO_ATTEMPT,
            "delay_submit_for_wiki_cooldown": DELAY_SUBMIT_FOR_WIKI_COOLDOWN,
            "target_submit_elapsed_seconds": TARGET_SUBMIT_ELAPSED_SECONDS,
            "min_seconds_left_at_submit": MIN_SECONDS_LEFT_AT_SUBMIT,
            "max_submit_wait_seconds": MAX_SUBMIT_WAIT_SECONDS,
            "answer_model": ANSWER_MODEL_FOR_GAME,
            "submit_answers": SUBMIT_ANSWERS,
        },
        "questions": [],
    }

    print(f"Started game session {game.session_id}. Competition {COMPETITION_ID}.")
    question_count = 0
    correct_count = 0

    while game.in_progress:
        question = game.current_question
        if question is None:
            print("No active question returned by server.")
            break

        # Do not reset the Wikipedia cooldown here; the server rate limit continues across questions.

        question_count += 1
        current_level = game.current_level
        time_left = seconds_available(game)

        print("\n" + "=" * 80)
        print(f"Question {question_count} | Level {current_level} | {time_left:.1f}s left")
        print(question.text)
        for index, opt in enumerate(question.options):
            print(f"  {LETTERS[index]}. [{option_id(opt)}] {option_text(opt)}")
        print("=" * 80)

        if time_left < MIN_SECONDS_TO_ATTEMPT:
            selected = fallback_option(question)
            prediction = {
                "question": question.text,
                "options": [{"id": option_id(opt), "text": option_text(opt)} for opt in question.options],
                "documents": [],
                "rag_answer": "Skipped RAG because not enough time remained.",
                "rag_method": "fallback_time_guard",
                "evidence_chunks": [],
                "option_match": {
                    "answer_id": option_id(selected),
                    "answer_text": option_text(selected),
                    "answer_index": 0,
                    "letter": "A",
                    "model_output": "fallback_time_guard",
                },
                "elapsed_seconds": 0.0,
                "seconds_left_start": time_left,
                "seconds_left_end": time_left,
            }
        else:
            prediction = answer_one_question_with_pipeline(question, game=game)

        selected_id = prediction["option_match"]["answer_id"]
        selected_text = prediction["option_match"]["answer_text"]
        selected_letter = prediction["option_match"]["letter"]

        if VERBOSE:
            print("\nRAG answer:")
            print(prediction["rag_answer"])
            print("\nClosest option:", f"{selected_letter}. [{selected_id}] {selected_text}")
            print("Matcher output:", prediction["option_match"].get("model_output"))
            timings = prediction.get("timings", {})
            if timings:
                print(
                    "Timing:",
                    f"direct={timings.get('direct_model_seconds', 0):.2f}s",
                    f"wiki={timings.get('wikipedia_seconds', 0):.2f}s",
                    f"rag={timings.get('rag_generation_seconds', 0):.2f}s",
                    f"match={timings.get('option_matching_seconds', 0):.2f}s",
                    f"total={timings.get('pre_submit_pipeline_seconds', prediction['elapsed_seconds']):.2f}s",
                )
            print(f"Elapsed: {prediction['elapsed_seconds']:.2f}s | Time left now: {seconds_available(game):.1f}s")

        result_payload = None
        if SUBMIT_ANSWERS:
            submission_wait_seconds = wait_before_submit_for_cooldown(game)
            prediction["submission_wait_seconds"] = submission_wait_seconds
            if submission_wait_seconds:
                print(f"Time left after cooldown wait: {seconds_available(game):.1f}s")

            if seconds_available(game) <= QUESTION_TIME_BUFFER:
                print("Warning: low time before submit; submitting selected option immediately.")
            result = game.answer(selected_id)
            result_payload = {
                "correct": result.correct,
                "timed_out": result.timed_out,
                "game_over": result.game_over,
                "earned_amount": result.earned_amount,
            }
            correct_count += int(bool(result.correct))

            if result.correct:
                print(f"Correct. Earned: {result.earned_amount}")
            elif result.timed_out:
                print(f"Timed out. Earned: {result.earned_amount}")
            else:
                print(f"Wrong. Earned: {result.earned_amount}")
        else:
            print("Dry run: answer not submitted.")

        run_log["questions"].append(
            {
                "number": question_count,
                "level": current_level,
                "prediction": prediction,
                "result": result_payload,
                "timestamp": datetime.now(timezone.utc).isoformat(),
            }
        )

        if result_payload and result_payload.get("game_over"):
            break
        if MAX_QUESTIONS is not None and question_count >= MAX_QUESTIONS:
            print("MAX_QUESTIONS reached; stopping.")
            break
        if not SUBMIT_ANSWERS:
            break

    run_log["finished_at"] = datetime.now(timezone.utc).isoformat()
    run_log["questions_answered"] = question_count
    run_log["correct_count"] = correct_count
    run_log["final_earned_amount"] = game.earned_amount

    if SAVE_RUN_LOG:
        log_dir = Path(RUN_LOG_DIR)
        log_dir.mkdir(parents=True, exist_ok=True)
        log_path = log_dir / f"test3_rag_game_{game.session_id}.json"
        with open(log_path, "w", encoding="utf-8") as handle:
            json.dump(run_log, handle, indent=2, ensure_ascii=False)
        print("Run log saved to:", log_path)

    print("\nGame summary")
    print("Questions answered:", question_count)
    print("Correct answers:", correct_count)
    print("Final earnings:", game.earned_amount)
    return game, run_log


final_game, final_run_log = play_actual_rag_game()

Preloading Qwen/Qwen2.5-7B-Instruct before starting timed game...
Loaded model: Qwen/Qwen2.5-7B-Instruct | device=cuda:0 | dtype=torch.bfloat16
Direct model ready in 0.5s. Starting game only after this point.
Preloading sentence embedding model sentence-transformers/all-MiniLM-L6-v2...
Sentence embedding reranker ready.
Preloading cross-encoder reranker cross-encoder/ms-marco-MiniLM-L-6-v2...
Cross-encoder reranker ready.
Started game session 336847. Competition 1.

Question 1 | Level 1 | 30.0s left
How did the Macedonian elite bury their rulers, according to archaeological evidence?
  A. [0] They constructed lavish tombs filled with grave goods and artwork.
  B. [1] They buried their rulers in simple pits without any grave goods.
  C. [2] They held elaborate funerals with cremation and burial of ashes.
  D. [3] They built large temples to honor the deceased.

RAG answer:
Tool router skipped live Wikipedia (high_direct_margin:15.542>=1.250); used direct Qwen/Qwen2.5-7B-Instruct answer.

---
## 5. News Pipeline
_Competition ID: 5 (News & Current Events)_

Techniques: Serper Google News API â†’ article scraping, Bing RSS + FAISS semantic
fallback, LLM-generated search keywords, date-window filtering.


### 5a. News Retrieval Helpers (Serper, Bing RSS, FAISS)

In [34]:
import concurrent.futures
import json
import builtins
import os
import re
import textwrap
import urllib.parse
import warnings
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta

import faiss
import requests
import trafilatura
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer


# Keep notebook output clean while the helper libraries load and run.
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.simplefilter(action="ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore")


SERPER_API_KEY = os.getenv("SERPER_API_KEY", "efc501e31e6f819db6f5c2546f86f9c239aebade")
USE_SEARCH_DATE_FILTER = True
SEARCH_DATE_INTERVAL_DAYS = 2


REQUEST_HEADERS = {
    "User-Agent": "Chrome/120.0",
    "Accept": "text/html",
    "Accept-Language": "en-US,en;q=0.5",
    "Referer": "https://www.google.com/",
    "DNT": "1",
    "Upgrade-Insecure-Requests": "1",
}


def clean_bing_redirect(url):
    """Return the real page URL when Bing wraps it in a redirect link."""
    if "bing.com" not in url or "url=" not in url.lower():
        return url

    parsed_url = urllib.parse.urlparse(url)
    query_args = urllib.parse.parse_qs(parsed_url.query)
    return query_args.get("url", [url])[0]


def read_article_body(url):
    """Try to pull readable article text from a URL."""
    resolved_url = clean_bing_redirect(url)

    # First try trafilatura because it usually strips menus, ads, and sidebars cleanly.
    try:
        html = trafilatura.fetch_url(resolved_url)
        if html:
            article_text = trafilatura.extract(html)
            if article_text and len(article_text) > 200:
                return article_text[:8000]
    except Exception:
        pass

    # If extraction fails, fall back to a simple BeautifulSoup paragraph scrape.
    try:
        response = requests.get(resolved_url, headers=REQUEST_HEADERS, timeout=4, allow_redirects=True)
        if response.status_code != 200:
            return ""

        soup = BeautifulSoup(response.text, "html.parser")
        for node in soup(["script", "style", "nav", "header", "footer", "aside"]):
            node.extract()

        useful_lines = []
        for paragraph in soup.find_all(["p", "li"]):
            line = paragraph.get_text(strip=True)
            if len(line) > 30:
                useful_lines.append(line)

        return " ".join(useful_lines)[:8000]
    except Exception:
        return ""


def make_date_filter(question_text):
    """Build a Serper date filter around a YYYY-MM-DD date found in the question."""
    match = re.search(r"\b(202\d)-(\d{2})-(\d{2})\b", question_text)
    if not match:
        return ""

    year, month, day = match.groups()
    try:
        target_day = datetime.strptime(f"{year}-{month}-{day}", "%Y-%m-%d")
        span = max(0, int(SEARCH_DATE_INTERVAL_DAYS))
        start_date = (target_day - timedelta(days=span)).strftime("%m/%d/%Y")
        end_date = (target_day + timedelta(days=span)).strftime("%m/%d/%Y")
        return f"cdr:1,cd_min:{start_date},cd_max:{end_date}"
    except Exception:
        return ""


def describe_date_window(date_window):
    match = re.search(r"cd_min:([^,]+),cd_max:([^,]+)", date_window or "")
    if not match:
        return ""
    return f"{match.group(1)} to {match.group(2)}"


def gather_primary_news(search_text, date_window=""):
    print(f"\nQuery trace: Search phrase: '{search_text}'")
    if date_window:
        print(f"        Date interval: {describe_date_window(date_window)}")

    if not SERPER_API_KEY:
        print("Query trace: SERPER_API_KEY not found - skipping primary lookup.")
        return ""

    request_body = {"q": search_text, "num": 6}
    if date_window:
        request_body["tbs"] = date_window

    headers = {"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"}

    try:
        response = requests.post(
            "https://google.serper.dev/news",
            headers=headers,
            data=json.dumps(request_body),
            timeout=10,
        )
        if response.status_code != 200:
            return ""

        results = response.json().get("news", [])

        # If the date-filtered query is too narrow, retry once without the date window.
        if not results and date_window:
            print("        Query trace: Date interval search empty; retrying broad search.")
            request_body.pop("tbs", None)
            wide_response = requests.post(
                "https://google.serper.dev/news",
                headers=headers,
                data=json.dumps(request_body),
                timeout=10,
            )
            results = wide_response.json().get("news", [])

        # News search can miss fresh pages, so use regular web search as one more fallback.
        if not results:
            print("        Query trace: News channel empty; checking general web results.")
            web_response = requests.post(
                "https://google.serper.dev/search",
                headers=headers,
                data=json.dumps(request_body),
                timeout=10,
            )
            results = web_response.json().get("organic", [])

        top_links = [item.get("link") for item in results[:2] if item.get("link")]
        article_text_by_url = {}

        with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
            futures = {executor.submit(read_article_body, link): link for link in top_links}
            for future in concurrent.futures.as_completed(futures):
                link = futures[future]
                try:
                    article_text_by_url[link] = future.result()
                except Exception:
                    article_text_by_url[link] = ""

        context_parts = []
        for item in results[:4]:
            title = item.get("title", "")
            date_label = item.get("date", "")
            snippet = item.get("snippet", "")
            link = item.get("link", "")

            block = f"Title: {title}\nDate: {date_label}\nSummary: {snippet}\n"
            if article_text_by_url.get(link):
                block += f"EXTENDED FULL TEXT: {article_text_by_url[link][:2500]}...\n"
            context_parts.append(block)

        return "\n".join(context_parts)
    except Exception as error:
        print(f"Query trace: Primary lookup error: {error}")
        return ""


class SecondaryNewsIndex:
    """Small Bing RSS + FAISS fallback used when Serper does not find enough evidence."""

    def __init__(self):
        print("Secondary index: Loading Bing RSS + vector fallback...")
        if "load_sentence_embedding_model" in globals():
            self.encoder = load_sentence_embedding_model()
        else:
            sentence_model_id = globals().get("SENTENCE_EMBEDDING_MODEL_ID", "sentence-transformers/all-MiniLM-L6-v2")
            cache = globals().setdefault("_SENTENCE_EMBEDDING_CACHE", {})
            if sentence_model_id not in cache:
                cache[sentence_model_id] = SentenceTransformer(sentence_model_id)
            self.encoder = cache[sentence_model_id]
        self.rss_cache = {}

    def make_chunks(self, text, chunk_size=600, overlap=150):
        clean_text = re.sub(r"\s+", " ", text)
        sentences = re.split(r"(?<=[.])\s+", clean_text)

        chunks = []
        current_chunk = ""
        for sentence in sentences:
            if len(current_chunk) + len(sentence) <= chunk_size:
                current_chunk += " " + sentence
            else:
                if current_chunk.strip():
                    chunks.append(current_chunk.strip())
                current_chunk = sentence

        if current_chunk.strip():
            chunks.append(current_chunk.strip())
        return chunks

    def search_backup_index(self, search_terms, semantic_query, top_k=6):
        search_words = search_terms.split()
        search_variants = [search_terms]
        if len(search_words) > 3:
            search_variants.append(" ".join(search_words[:-1]))
        if len(search_words) > 2:
            search_variants.append(" ".join(search_words[:2]))

        candidate_chunks = []
        seen_urls = set()
        headers = {"User-Agent": "Mozilla/5.0"}

        for query in search_variants:
            if not query.strip():
                continue

            if query in self.rss_cache:
                candidate_chunks.extend(self.rss_cache[query])
                break

            chunks_for_query = []
            try:
                encoded_query = urllib.parse.quote(query)
                rss_url = f"https://www.bing.com/news/search?q={encoded_query}&format=rss"
                response = requests.get(rss_url, headers=headers, timeout=10)
                if response.status_code != 200:
                    continue

                root = ET.fromstring(response.text)
                items = root.findall(".//channel/item")

                feed_items = []
                for item in items:
                    link_node = item.find("link")
                    link = link_node.text if link_node is not None else ""
                    if link and link not in seen_urls:
                        seen_urls.add(link)
                        feed_items.append((item, link))
                    if len(feed_items) >= 3:
                        break

                def read_feed_item(feed_item):
                    item, link = feed_item
                    title_node = item.find("title")
                    desc_node = item.find("description")

                    title = title_node.text if title_node is not None else ""
                    description = desc_node.text if desc_node is not None else ""
                    description = re.sub("<[^<]+>", " ", description)

                    article_text = read_article_body(link)
                    combined_text = f"{title}. {description}. {article_text}"
                    return self.make_chunks(combined_text)

                with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
                    for chunks in executor.map(read_feed_item, feed_items):
                        chunks_for_query.extend(chunks or [])

                self.rss_cache[query] = chunks_for_query
                candidate_chunks.extend(chunks_for_query)
                if candidate_chunks:
                    break
            except Exception:
                continue

        if not candidate_chunks:
            return ""

        # Remove near-duplicate chunks before embedding them.
        deduped_chunks = []
        chunk_signatures = set()
        for chunk in candidate_chunks:
            signature = chunk[:100].strip()
            if signature not in chunk_signatures:
                chunk_signatures.add(signature)
                deduped_chunks.append(chunk)

        if not deduped_chunks:
            return ""

        embeddings = self.encoder.encode(deduped_chunks, convert_to_numpy=True)
        index = faiss.IndexFlatL2(embeddings.shape[1])
        index.add(embeddings)

        question_vector = self.encoder.encode([semantic_query], convert_to_numpy=True)
        _, nearest_ids = index.search(question_vector, min(top_k, len(deduped_chunks)))

        selected_chunks = [deduped_chunks[index_id] for index_id in nearest_ids[0][:6]]
        return "\n\n".join(selected_chunks)


secondary_news_index = SecondaryNewsIndex()


def wrap_output_text(text, indent="", subsequent_indent=None):
    subsequent_indent = indent if subsequent_indent is None else subsequent_indent
    return textwrap.fill(
        str(text or ""),
        width=100,
        initial_indent=indent,
        subsequent_indent=subsequent_indent,
        break_long_words=False,
        break_on_hyphens=False,
    )


def display_question(question, question_count, level, seconds_left=None):
    border = "=" * 80
    time_label = "" if seconds_left is None else f" | {seconds_left:.1f}s left"

    print("\n" + border)
    print(f"Question {question_count} | Level {level}{time_label}")
    print(wrap_output_text(question.text))

    for index, option in enumerate(question.options):
        answer_letter = chr(65 + index)
        line = f"{answer_letter}: {option.id}: {option.text}"
        print(wrap_output_text(line, indent="  ", subsequent_indent="     "))

    print(border)


def pull_eval_summary(text):
    match = re.search(r"Eval:\s*(.+?)(?:\n|FINAL ANSWER:|$)", str(text or ""), re.IGNORECASE | re.DOTALL)
    if not match:
        return ""
    return re.sub(r"\s+", " ", match.group(1)).strip()


def make_search_keywords(question_text, options):
    prompt = f"""[INST] You are a specialist in news-search query construction.
Create a focused Google News query of no more than 5 words that can locate the exact article.

Rules:
1. Use only the core event, distinctive proper nouns, and main subjects from the question.
2. Do not include or borrow wording from the answer choices.
3. Return bare noun keywords separated by spaces. No verbs, punctuation, or explanation.

Generate keywords for this question:
{question_text}
Keywords: [/INST]"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=25,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    keywords = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    keywords = re.sub(r"\[/?INST\]|Keywords?:", " ", keywords, flags=re.IGNORECASE)
    keywords = re.sub(r"[^\w\s\-']", " ", keywords)
    return re.sub(r"\s+", " ", keywords).strip()


def read_final_letter(text):
    upper_text = text.upper()
    if "FINAL ANSWER: NONE" in upper_text:
        return "N"

    match = re.search(r"FINAL ANSWER:\s*([ABCD])", text, re.IGNORECASE)
    if match:
        return match.group(1).upper()

    letters = re.findall(r"\b([ABCD])\b", upper_text)
    return letters[-1] if letters else "A"


def ask_news_model(context, question):
    prompt = f"""[INST] You are a careful current-events analyst answering a multiple-choice question from retrieved news context.

Context:
{context}

Question:
{question.text}
A) {question.options[0].text}
B) {question.options[1].text}
C) {question.options[2].text}
D) {question.options[3].text}

Instructions:
1. Choose the option best supported by the evidence.
2. If the question asks for what is NOT, EXCEPT, false, missing, or denied, choose the option that the text excludes or fails to support.
3. If several options appear as nested locations or entities, choose the most specific one.
4. Combine details across snippets when that is necessary to identify the correct option.
5. If the context does not support an answer, output 'FINAL ANSWER: NONE'. Do not guess.
6. After 'FINAL ANSWER:' output only A, B, C, D, or NONE. Do not write the option text.

Use exactly this format:
Eval: justify the selected option in at most 15 words
FINAL ANSWER: A, B, C, D, or NONE
[/INST]Eval: """

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3072).to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    reply = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    reply = re.sub(r"\[/?INST\]", " ", reply, flags=re.IGNORECASE)
    reply = re.sub(r"\s+", " ", reply).strip()
    return re.sub(r"\s*FINAL ANSWER:", "\nFINAL ANSWER:", reply, flags=re.IGNORECASE).strip()


def select_answer(question):
    if len(question.options) < 4:
        return question.options[0].id, "A", "fallback"

    search_terms = make_search_keywords(question.text, question.options)
    date_filter = make_date_filter(question.text) if USE_SEARCH_DATE_FILTER else ""

    answer_letter = "N"
    model_reply = ""

    # Primary path: look for direct news evidence first.
    context = gather_primary_news(search_terms, date_filter)
    if context.strip():
        print("\n" + "-" * 40)
        print("Source packet A: Primary Serper evidence:")
        print(context)
        print("-" * 40 + "\n")

        model_reply = ask_news_model(context, question)
        answer_letter = read_final_letter(model_reply)
        print(f"Model read A:\n{model_reply}")

    # Secondary path: if the model says NONE, gather RSS evidence and rank it semantically.
    if answer_letter == "N":
        print("\nSecondary pass: Primary answer was NONE; checking fallback evidence...")
        option_text = " ".join(option.text for option in question.options)
        semantic_query = f"{question.text} {option_text}"
        backup_evidence = secondary_news_index.search_backup_index(
            search_terms=search_terms,
            semantic_query=semantic_query,
            top_k=6,
        )

        if backup_evidence.strip():
            print("\n" + "-" * 40)
            print("Source packet B: Bing RSS fallback evidence:")
            print(backup_evidence)
            print("-" * 40 + "\n")

            model_reply = ask_news_model(backup_evidence, question)
            answer_letter = read_final_letter(model_reply)
            print(f"Model read B:\n{model_reply}")
        else:
            print("\nSecondary pass: No fallback evidence found.")

    # Keep the caller from crashing when neither retrieval path supports an answer.
    if answer_letter == "N":
        print("\nFinal fallback: No supported answer found; using option A.")
        answer_letter = "A"

    try:
        choice_index = ["A", "B", "C", "D"].index(answer_letter)
    except ValueError:
        choice_index = 0
        answer_letter = "A"

    return question.options[choice_index].id, answer_letter, model_reply

Secondary index: Loading Bing RSS + vector fallback...


### 5b. News Game Loop

In [35]:
from millionaire_client import MillionaireClient
from millionaire_client.exceptions import TimeoutError, RateLimitError


def play_game(competition_id=5, mode='text'):
    API_URL = 'http://131.175.15.22:51111/'
    client = MillionaireClient(API_URL)
    user = client.login('gary', '13790229')
    print(f"Session user: {user.username}")



    game = client.game.start(competition_id=competition_id, mode=mode)
    question_count = 0
    correct_answers = 0

    while game.in_progress:
        question = game.current_question
        if question is None:
            break

        question_count += 1
        seconds_left = getattr(game, "time_remaining", None)
        display_question(question, question_count, game.current_level, seconds_left)

        t0 = time.time()
        option_id, answer_letter, answer_trace = select_answer(question)
        t1 = time.time()

        chosen_offset = builtins.max(0, ord(answer_letter) - ord("A")) if answer_letter in "ABCD" else 0
        chosen_text = question.options[chosen_offset].text if chosen_offset < len(question.options) else ""
        print("-" * 80)
        print(wrap_output_text(f"Selected answer: {answer_letter}: {option_id}: {chosen_text}"))
        brief_note = pull_eval_summary(answer_trace)
        if brief_note:
            print(wrap_output_text(f"Brief rationale: {brief_note}"))
        print(f"Processing time: {t1-t0:.2f}s")

        try:
            result = game.answer(option_id)
            if result.correct:
                correct_answers += 1
            print(f"Outcome: {'CORRECT' if result.correct else 'WRONG'} | Correct answers: {correct_answers}")
        except TimeoutError:
            print("Outcome: TIMED OUT | generation took more than thirty seconds")
            break
        except RateLimitError:
            print("Rate limited; waiting five seconds before retrying submit.")
            time.sleep(5)
            result = game.answer(option_id)
            if result.correct:
                correct_answers += 1
            print(f"Outcome: {'CORRECT' if result.correct else 'WRONG'} | Correct answers: {correct_answers}")

        if result.game_over:
            break
        time.sleep(1)

    print(f"Final correct answers: {correct_answers}")
    game.correct_answers = correct_answers
    return game

### 5c. Run News Game

In [40]:
game = play_game(competition_id=5, mode='text')

Session user: gary

Question 1 | Level 1 | 30.0s left
What event occurred on 18 March that had a severe impact on Gulf economies, as reported on
2026-05-06?
  A: 0: The signing of a peace treaty
  B: 1: A natural gas discovery
  C: 2: A political coup
  D: 3: An attack on Qatar's LNG facilities

Query trace: Search phrase: 'Gulf Economies Event 18March2026'
        Date interval: 05/04/2026 to 05/08/2026
        Query trace: Date interval search empty; retrying broad search.


ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.abs-cbn.com/news/business/2026/3/20/middle-east-war-global-economic-fallout-1336
ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None



----------------------------------------
Source packet A: Primary Serper evidence:
Title: Middle East war: global economic fallout
Date: Mar 20, 2026
Summary: Here are the latest economic events in the Middle East war.

Title: Middle East war: global economic fallout
Date: Mar 18, 2026
Summary: Photo: AFP. Here are the latest economic events in the Middle East war on Tuesday: STOCKS RISE AS OIL PUSHES HIGHER...
EXTENDED FULL TEXT: NewsPoliticsGovernanceCrime and JusticeAccidents and FiresTechnologyEducationEnvironmentHealthcareWorld OpinionEditorialViewsInterviews BusinessEconomyAgricultureIndustryStartupsGlobal Economy SportsCricketFootballMore SportsTennis LifestyleFashionRelationshipsHeath and WellnessFood and RecipeTravelogue CultureEntertainmentBooks and LiteratureHeritageTv & FilmMusicTheatre & Arts Slow ReadsIn FocusGeopolitical InsightsBig PictureUnheard Voices YouthAcademicsCareer and SkillsCampus LifeOff CampusPop Culture Ds+Business +Investigative StoriesRoundtablesSuppleme

ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None



----------------------------------------
Source packet A: Primary Serper evidence:
Title: Police raid neo-nazi networks across Germany
Date: 3 weeks ago
Summary: Deutsche Jugend Voran (Forwards German Youth) marching in 2024 German police swooped on addresses across a dozen federal states on 6 May, targeting suspecte...
EXTENDED FULL TEXT: German police swooped on addresses across a dozen federal states on 6 May, targeting suspected ringleaders of two neo-nazi youth networks whose combined membership runs into the hundreds. Around fifty homes were searched in raids stretching from Berlin to Bavaria, though no arrests followed. The two groups in the crosshairs were Deutsche Jugend Voran (Forwards German Youth) and Jung und Stark (Young and Strong). As these groups advocate for using violence against political opponents, they are being treated as criminal organisations. In some cases, this was not just an empty threat; the Federal Prosecutor’s Office said, ‘Some of the accused are alleg

...

ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.nytimes.com/2026/05/14/world/americas/cuba-oil-energy-crisis.html



----------------------------------------
Source packet A: Primary Serper evidence:
Title: CIA chief visits Cuba as energy crisis worsens
Date: 2 weeks ago
Summary: The reported visit to Havana came after the US renewed an offer of aid to ease the effects of its oil blockade.
EXTENDED FULL TEXT: CIA chief visits Cuba as energy crisis worsens
CIA director John Ratcliffe has met his Cuban counterpart at the interior ministry in Havana, after the US renewed an offer of $100m (£74m) of aid to ease the effects of its oil blockade.
A Cuban statement said the meeting was an attempt to improve dialogue and American officials were told Havana was not a threat to US national security.
The development comes as the United States government is preparing to bring charges against Raúl Castro for the downing of two small planes 30 years ago, according to CBS, the BBC's US partner.
Fuel shortages exacerbated by the US oil blockade on the country have left hospitals unable to function normally and force

ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None



----------------------------------------
Source packet A: Primary Serper evidence:
Title: Japan is calling. Play Forza Horizon 6 now, and speed ... - Facebook
Date: May 20, 2026
Summary: 🏎️ Playground Games just confirmed the series' most ambitious map ever, with a full Japan recreation featuring Tokyo, Mount Fuji, rural mountain ...

Title: Forza Horizon 6 is gaining attention after reports ... - Facebook
Date: May 19, 2026
Summary: Communities are discussing how making these elements indestructible allows the game to maintain authenticity and respect while still delivering ...

Title: Forza Horizon 6: More Details on The Country, The ... - XBOX Wire
Date: Jan 22, 2026
Summary: At today's Developer_Direct, we got an extended look at Forza Horizon 6, brought to life in the stunning landscapes of Japan.

Title: Forza Horizon 6 Brings Authentic Japanese Racing Culture After ...
Date: May 18, 2026
Summary: Playground Games brings authentic Japanese racing culture to Forza Horizon 6 with 


----------------------------------------
Source packet B: Bing RSS fallback evidence:
Forza Horizon 6 review: Playground Games finally delivers the Japan that Forza fans have always wanted. Japan has been at the top of the Forza Horizon community's wishlist since the series launched in Colorado fourteen years ago. Not just on the wishlist—at the very top of it, year after year, with a ....

'Forza Horizon 6' review: Japan was worth the wait. The menu screen loads, and before you've touched the throttle, Forza Horizon 6 has already made its first impression. A track called You by Kasablanca and Lane 8 plays out, and if the YouTube comments .... 'Forza Horizon 6' review: Japan was worth the wait Playground Games' new chapter in the racing game franchise turns you into the ultimate tourist driver in Japan.

Discover stunning landscapes, from Shibuya's neon streets to cherry blossom mountains - PUBLISHED: Fri 15 May 2026, 7:27 PM The menu screen loads, and before you've touched the thrott

---
## 6. Maths Pipeline
_Competition ID: 3 (Maths)_

Techniques:  


In [ ]:
#TODO

---
## 7. Evaluation & Analysis
_Run after collecting logs from any or all pipelines above._


In [41]:
# Compile all results into a DataFrame
def log_to_df(log, model_name, competition_id):
    rows = []
    for i, entry in enumerate(log):
        rows.append({
            "model": model_name,
            "competition_id": competition_id,
            "level": entry.get("level", i+1),
            "correct": entry.get("correct", False),
            "timed_out": entry.get("timed_out", False),
            "elapsed_s": entry.get("elapsed", 0),
            "earned": entry.get("earned", 0),
            "question": entry.get("question", ""),
        })
    return pd.DataFrame(rows)

# Combine all logs
all_dfs = []
for log, model_name, comp_id in [
    (baseline_log, "Zero-Shot", COMP_ID),
    (fewshot_log,  "Few-Shot",  COMP_ID),
    (rag_log,      "RAG",       1),
    (maths_log,    "Model + SymPy",   3),
    (ensemble_log, "Ensemble",           COMP_ID),
]:
    if log:
        all_dfs.append(log_to_df(log, model_name, comp_id))

df = pd.concat(all_dfs, ignore_index=True)
print(f"Total game entries logged: {len(df)}")
df.head(10)

NameError: name 'maths_log' is not defined

In [ ]:
# Summary statistics by model, compute we shall
summary = df.groupby("model").agg(
    questions_answered=("correct", "count"),
    accuracy=("correct", lambda x: x.dropna().mean()),
    avg_response_time=("elapsed_s", "mean"),
    timeouts=("timed_out", "sum"),
    max_earned=("earned", "max"),
).round(3)

print("=== Model Performance Summary ===")
print(summary.to_string())

In [ ]:
# Plot: Accuracy by Model
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Accuracy bar chart
models = summary.index.tolist()
accs = summary["accuracy"].tolist()
axes[0].bar(models, accs, color=["#4CAF50","#2196F3","#FF9800","#9C27B0","#F44336"][:len(models)])
axes[0].set_title("Accuracy by Model")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=30)

# Response time
times = summary["avg_response_time"].tolist()
axes[1].bar(models, times, color="#607D8B")
axes[1].set_title("Avg Response Time (s)")
axes[1].set_ylabel("Seconds")
axes[1].tick_params(axis="x", rotation=30)

# Earnings
earned = summary["max_earned"].tolist()
axes[2].bar(models, earned, color="#FF5722")
axes[2].set_title("Max Earned ($)")
axes[2].set_ylabel("USD")
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150)
plt.show()
print("Plot saved, it has been.")

In [ ]:
# Accuracy by difficulty level â€” harder questions, worse models do?
level_acc = df.groupby("level")["correct"].mean().reset_index()
level_acc.columns = ["Level", "Accuracy"]

plt.figure(figsize=(10, 4))
plt.plot(level_acc["Level"], level_acc["Accuracy"], marker="o", color="#2196F3", linewidth=2)
plt.axhline(0.25, linestyle="--", color="red", label="Random chance (25%)")
plt.fill_between(level_acc["Level"], level_acc["Accuracy"], 0.25,
                 where=level_acc["Accuracy"] > 0.25, alpha=0.2, color="green", label="Above chance")
plt.title("Accuracy by Question Level (All Models)")
plt.xlabel("Level")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig("accuracy_by_level.png", dpi=150)
plt.show()

In [ ]:
# Response time distribution â€” within 30s, we must stay!
plt.figure(figsize=(10, 4))
for model_name, grp in df.groupby("model"):
    plt.hist(grp["elapsed_s"], bins=15, alpha=0.5, label=model_name)
plt.axvline(30, color="red", linestyle="--", label="30s timeout")
plt.title("Response Time Distribution by Model")
plt.xlabel("Seconds")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.savefig("response_times.png", dpi=150)
plt.show()

too_slow = df[df["elapsed_s"] > 25]
print(f"Responses dangerously close to timeout (>25s): {len(too_slow)}")

## 7. Research Questions Analysis

In [ ]:
# Q1: Are some models better at certain topics than others?
# Explore, we must â€” competition_id = category proxy
cat_map = {0: "Entertainment", 1: "Ancient History", 2: "Science", 3: "Maths"}

cat_acc = df.groupby(["model","competition_id"])["correct"].mean().reset_index()
cat_acc["category"] = cat_acc["competition_id"].map(cat_map)

pivot = cat_acc.pivot(index="model", columns="category", values="correct")
print("=== Accuracy by Model and Category ===")
print(pivot.round(2).to_string())

In [ ]:
# Q2: Is the model overconfident? (can check via prompt output entropy â€” rough approximation)
# Q3: What types of questions do models struggle on?
wrong_qs = df[df["correct"] == False][["model","level","question"]].dropna()
print(f"=== Sample of Wrong Answers ({len(wrong_qs)} total) ===")
print(wrong_qs.head(10).to_string(index=False))

In [ ]:
# Q4: Does RAG help vs. not?
# Compare Flan-T5 Zero-Shot vs Flan-T5 RAG on the same competition (if available)
rag_comp = df[df["model"].isin(["Flan-T5 Zero-Shot", "Flan-T5 RAG"])]
if len(rag_comp) > 0:
    comparison = rag_comp.groupby("model")["correct"].mean()
    print("=== RAG vs. No RAG ===")
    print(comparison)
    improvement = comparison.get("Flan-T5 RAG", 0) - comparison.get("Flan-T5 Zero-Shot", 0)
    print(f"RAG improvement: {improvement:+.1%}")

---
## 8. Best System â€” Final Run

Update `best_strategy` below based on your evaluation results, then run all
competitions in one go.


In [ ]:
# Best strategy, define we must based on evaluation results above
# Update this after running all experiments!

def best_strategy(question):
    """
    Winning strategy, this is. Combine RAG + few-shot + math tool, we do.
    - Maths questions â†’ SymPy tool
    - Other questions â†’ RAG + few-shot prompt
    """
    question_text_lower = question.text.lower()

    # Math detection heuristic
    math_keywords = ["calculate","compute","solve","equation","percentage","%" ,"sum",
                     "product","divided","multiplied","squared","factorial","derivative"]
    is_math = any(kw in question_text_lower for kw in math_keywords)

    if is_math:
        return answer_maths(question)
    else:
        return answer_with_rag(question)

print("Best strategy ready, it is. To the leaderboard, go we shall!")

In [ ]:
# Play all 4 competitions with the best strategy, we shall
comp_names = {0: "Entertainment", 1: "Ancient History", 2: "Science & Nature", 3: "Maths"}
final_results = {}

for comp_id in [0, 1, 2, 3]:
    print(f"\n{'='*60}")
    print(f"Playing: {comp_names[comp_id]}")
    print(f"{'='*60}")
    log, level, earned = play_full_game(
        competition_id=comp_id,
        answer_fn=best_strategy,
        label=f"Best System â€” {comp_names[comp_id]}"
    )
    final_results[comp_id] = {"level": level, "earned": earned}
    time.sleep(2)  # Be polite to the server, we must

print("\n=== FINAL RESULTS ===")
for cid, res in final_results.items():
    print(f"  {comp_names[cid]}: Level {res['level']} | ${res['earned']:,.0f}")

In [ ]:
# Check leaderboard positions, we shall
print("=== Leaderboard Positions ===")
for comp_id in [0, 1, 2, 3]:
    lb = client.leaderboard.get(competition_id=comp_id, limit=20)
    print(f"\n--- {lb.competition.name} ---")
    for i, entry in enumerate(lb.entries[:10], 1):
        marker = " â† YOU" if entry.username == USERNAME else ""
        print(f"  {i}. {entry.username}: ${entry.score:,.0f} (Level {entry.reached_level}){marker}")

## 11. Conclusions

### Key Findings

| Question | Finding |
|----------|---------|
| Zero-shot vs few-shot vs CoT? | *(fill in after experiments)* |
| Does RAG improve accuracy? | *(fill in after experiments)* |
| Does SymPy help for Maths? | *(fill in after experiments)* |
| Are bigger models better? | *(fill in after experiments)* |
| Can we answer within 30s? | *(fill in after experiments)* |
| Which category is hardest? | *(fill in after experiments)* |